# 华泰动量类因子

华泰动量类因子研报当中给出了四个效果较好的因子，并且给出了其构建方式，这四个因子分别是 exp_wgt_return_6m、exp_wgt_return_3m、wgt_return_1m、return_1m，它们的构造方式分别是：

-exp_wgt_return_6m=个股最近 6 个月内，以“每日换手率 × exp(-x_i/6/4)”作为权重，对每日收益率求加权平均得到的因子值，其中 x_i 为该日距离截面日的交易日天数，计算时排除非交易日。

-exp_wgt_return_3m=个股最近 3 个月内，以“每日换手率 × exp(-x_i/3/4)”作为权重，对每日收益率求加权平均得到的因子值，其中 x_i 为该日距离截面日的交易日天数，计算时排除非交易日。

-wgt_return_1m=个股最近 1 个月内，以每日换手率作为权重，对每日收益率求加权平均得到的因子值。

-return_1m=个股最近 1 个月的区间收益率，即最近 1 个月股价涨跌幅。

对于每一个因子，研报中显示，因子值越低的组表现的越好，同时除了return_1m是对于市值处于1/3~2/3之间的股票表现最好，其余的都是市值排名在后1/3的股票表现较好，且每个因子都有行业偏好，因此在设计策略时都是以先筛选市值，再筛选行业，最后取因子值低的股票进行等权调仓，以此作为因子的回测策略，接下来逐个因子进行复现。

## exp_wgt_return_6m 因子

先通过上述策略对因子的收益曲线进行回测，直观的感受该因子的效果

In [ ]:
import math
import hashlib
import pandas as pd
from datetime import datetime, timedelta

import dai
from bigquant import bigtrader


# =====================================================
# 1. 策略参数区
# =====================================================

# 回测区间
START_DATE = "2024-01-01"
END_DATE = "2026-06-01"

# 每次持有的股票数量
HOLD_NUM = 100

# 每隔多少个交易日调仓一次
REBALANCE_DAYS = 20

# 目标总仓位：不要满仓，避免现金不足、手续费、整手约束造成回测不稳定
TARGET_TOTAL_WEIGHT = 0.98

# exp_wgt_return_6m：6个月动量因子
FACTOR_MONTHS = 6

# 6个月近似为126个交易日
LOOKBACK_DAYS = 21 * FACTOR_MONTHS

# 为了计算 m_lag，需要在回测开始日前多取一段历史
BEFORE_START_DAYS = 300

# list_days 是自然日口径，不是交易日口径
# 126个交易日大约对应180个自然日，这里多留缓冲
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 初始资金
CAPITAL_BASE = 1_000_000

# 剔除行业：默认使用中信一级行业 cs_level1_name
EXCLUDE_INDUSTRIES = [
    "交通运输",
    "电力及公用事业",
    "纺织服装",
    "轻工制造",
    "家电",
    "石油石化",
    "综合",
    "银行",
]


# =====================================================
# 2. 构造 exp_wgt_return_6m 因子表达式
# =====================================================

def lag(field: str, k: int) -> str:
    """生成 BigQuant DAI SQL 的 m_lag 表达式。k=0 表示当前值。"""
    if k == 0:
        return field
    return f"m_lag({field}, {k})"


num_terms = []
den_terms = []

for i in range(LOOKBACK_DAYS):
    # 指数衰减权重：exp(-x_i / N / 4)
    # N = FACTOR_MONTHS = 6
    decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)

    close_i = lag("close", i)
    close_i_1 = lag("close", i + 1)
    turn_i = lag("turn", i)

    # 第 t-i 日收益率
    ret_i = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

    # 分子：收益率 * 换手率 * 指数衰减权重
    num_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i} * {ret_i}), 0.0)"
    )

    # 分母：换手率 * 指数衰减权重
    den_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)"
    )


num_expr = " + ".join(num_terms)
den_expr = " + ".join(den_terms)

industry_sql = ", ".join([f"'{x}'" for x in EXCLUDE_INDUSTRIES])

calc_start_date = (
    datetime.strptime(START_DATE, "%Y-%m-%d") - timedelta(days=BEFORE_START_DAYS)
).strftime("%Y-%m-%d")


# =====================================================
# 3. 用 DAI SQL 生成每日候选股票
# =====================================================

sql = f"""
WITH factor_raw AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        list_days,
        is_risk_warning,
        suspended,

        -- 市值从小到大做截面百分位排名
        -- mcap_pct <= 1/3 表示市值排名后1/3，即小市值股票
        c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,

        -- 华泰 exp_wgt_return_6m 因子
        ({num_expr}) / NULLIF(({den_expr}), 0) AS exp_wgt_return_6m

    FROM cn_stock_prefactors

    WHERE date >= '{calc_start_date}'
      AND date <= '{END_DATE}'

      -- 只取沪深A股，剔除北交所
      AND list_sector IN (1, 2, 3)

      -- 剔除ST / 风险警示
      AND is_risk_warning = 0

      -- 剔除停牌
      AND suspended = 0

      -- 剔除上市时间过短的股票
      AND list_days >= {MIN_LIST_DAYS}

      -- 基础字段非空
      AND close IS NOT NULL
      AND turn IS NOT NULL
      AND total_market_cap IS NOT NULL
      AND cs_level1_name IS NOT NULL
),

universe AS (
    SELECT
        *
    FROM factor_raw
    WHERE date >= '{START_DATE}'

      -- 市值排名后1/3
      AND mcap_pct <= 1.0 / 3.0

      -- 剔除指定中信一级行业
      AND cs_level1_name NOT IN ({industry_sql})

      -- 因子值非空
      AND exp_wgt_return_6m IS NOT NULL
),

ranked AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        exp_wgt_return_6m,

        -- 稳定排序：
        -- 1. 因子值从低到高；
        -- 2. 因子值相同时，市值从小到大；
        -- 3. 仍相同时，股票代码从小到大。
        row_number() OVER (
            PARTITION BY date
            ORDER BY exp_wgt_return_6m ASC, total_market_cap ASC, instrument ASC
        ) AS factor_rank

    FROM universe
)

SELECT
    date,
    instrument,
    cs_level1_name,
    total_market_cap,
    exp_wgt_return_6m,
    factor_rank

FROM ranked

-- 每日只保留因子值最低的 HOLD_NUM 只股票
WHERE factor_rank <= {HOLD_NUM}

ORDER BY date ASC, factor_rank ASC, instrument ASC
"""

signal_df = dai.query(sql).df()

signal_df["date"] = pd.to_datetime(signal_df["date"]).dt.strftime("%Y-%m-%d")

print("原始每日信号样例：")
print(signal_df.head())
print("原始信号日期数量：", signal_df["date"].nunique())
print("原始信号股票数量：", signal_df["instrument"].nunique())

if signal_df.empty:
    raise ValueError("signal_df 为空，请检查日期范围、行业过滤、市值过滤或字段名称。")


# =====================================================
# 4. 按 REBALANCE_DAYS 生成调仓日
# =====================================================

all_signal_dates = sorted(signal_df["date"].unique())

# 每隔 REBALANCE_DAYS 个交易日调仓一次
rebalance_dates = set(all_signal_dates[::REBALANCE_DAYS])

signal_df = signal_df[signal_df["date"].isin(rebalance_dates)].copy()

# 为确保后续交易顺序稳定，先排序
signal_df = signal_df.sort_values(
    ["date", "factor_rank", "instrument"]
).reset_index(drop=True)

# 每个调仓日等权持有 HOLD_NUM 只股票
# 使用 TARGET_TOTAL_WEIGHT，避免满仓
signal_df["weight"] = signal_df.groupby("date")["instrument"].transform(
    lambda x: TARGET_TOTAL_WEIGHT / len(x)
)

print("调仓信号样例：")
print(signal_df.head())
print("调仓次数：", signal_df["date"].nunique())
print("调仓后涉及股票数量：", signal_df["instrument"].nunique())


# =====================================================
# 5. 打印信号哈希，用来检查信号是否稳定
# =====================================================

check_df = signal_df[["date", "instrument", "factor_rank", "weight"]].copy()
check_df = check_df.sort_values(["date", "factor_rank", "instrument"])

signal_hash = hashlib.md5(
    check_df.to_csv(index=False).encode("utf-8")
).hexdigest()

print("信号哈希：", signal_hash)


# =====================================================
# 6. BigTrader 回测函数
# =====================================================

def initialize(context: bigtrader.IContext):
    context.signal_data = context.data.copy()
    context.signal_data["date"] = pd.to_datetime(
        context.signal_data["date"]
    ).dt.strftime("%Y-%m-%d")

    context.signal_data = context.signal_data.sort_values(
        ["date", "factor_rank", "instrument"]
    ).reset_index(drop=True)

    context.signal_dates = set(context.signal_data["date"].unique())

    # 手续费设置：
    # buy_cost：买入佣金
    # sell_cost：卖出佣金 + 印花税近似合计
    # min_cost：最低手续费
    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=0.0003,
            sell_cost=0.0013,
            min_cost=5.0
        )
    )


def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    today = data.current_dt.strftime("%Y-%m-%d")

    # 非调仓日不操作
    if today not in context.signal_dates:
        return

    today_df = context.signal_data[
        context.signal_data["date"] == today
    ].copy()

    if today_df.empty:
        return

    # 确保目标股票买入顺序稳定
    today_df = today_df.sort_values(
        ["factor_rank", "instrument"]
    ).reset_index(drop=True)

    target_instruments = set(today_df["instrument"].tolist())

    # 当前持仓
    current_positions = context.get_positions()
    holding_instruments = set(current_positions.keys())

    # 先卖出不在目标池里的股票
    # 注意：set 本身无序，所以这里必须 sorted
    sell_list = sorted(holding_instruments - target_instruments)

    for instrument in sell_list:
        context.order_target_percent(instrument, 0)

    # 再买入 / 调整目标股票到等权
    # today_df 已经按照 factor_rank + instrument 排序，保证顺序稳定
    for _, row in today_df.iterrows():
        instrument = row["instrument"]
        target_weight = float(row["weight"])

        context.order_target_percent(
            instrument,
            target_weight
        )


# =====================================================
# 7. 运行回测
# =====================================================

instruments = sorted(signal_df["instrument"].unique().tolist())

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,

    start_date=START_DATE,
    end_date=END_DATE,

    capital_base=CAPITAL_BASE,
    instruments=instruments,
    data=signal_df,

    initialize=initialize,
    handle_data=handle_data,

    benchmark="000300.SH",

    # T日产生信号，下一根K线开盘成交
    order_price_field_buy="open",
    order_price_field_sell="open",

    # 单只股票成交量限制
    # 如果你想进一步排查回测不稳定，可以临时改成 1.0
    volume_limit=0.025,
)



为了更加全面的了解这个因子具体的选股能力，以及其对收益率的解释作用，我们计算其IC值和用回归法计算其因子收益率，对于IC值与因子收益率，都同时考虑两种情况，一种情况是在不剔除行业和市值因素的，一版是剔除行业与市值影响的，在计算这些指标时都是在小市值股票池当中进行的计算

In [ ]:
import math
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import dai
import statsmodels.api as sm

from IPython.display import display


# =====================================================
# 1. 参数区
# =====================================================

START_DATE = "2024-01-01"
END_DATE = "2026-06-01"

# 每隔多少个交易日调仓一次
# 因子检验频率和未来收益周期都绑定这个变量
REBALANCE_DAYS = 20

# 未来收益周期绑定持仓周期
FORWARD_DAYS = REBALANCE_DAYS

# exp_wgt_return_6m：6个月动量因子
FACTOR_MONTHS = 6
LOOKBACK_DAYS = 21 * FACTOR_MONTHS

# 为了计算 m_lag，需要在分析开始日前多取一段历史
BEFORE_START_DAYS = 300

# 为了计算未来收益，需要在 END_DATE 后多取一段数据
AFTER_END_DAYS = int(FORWARD_DAYS * 365 / 252) + 30

# list_days 是自然日口径，不是交易日口径
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 每个调仓截面最少样本数
MIN_OBS = 50

# 去极值分位数
WINSOR_Q_LOW = 0.01
WINSOR_Q_HIGH = 0.99

# 是否使用你策略里的股票池约束
# True：只看小市值后1/3，并剔除指定行业
# False：看全A非ST、非停牌、上市足够久股票
USE_STRATEGY_UNIVERSE = True

# 你的策略是 ORDER BY exp_wgt_return_6m ASC
# 即因子值越低越好，所以方向设为 -1
# 如果未来换成“因子值越高越好”的因子，改成 1
FACTOR_DIRECTION = -1

# 剔除行业：沿用你策略里的中信一级行业字段 cs_level1_name
EXCLUDE_INDUSTRIES = [
    "交通运输",
    "电力及公用事业",
    "纺织服装",
    "轻工制造",
    "家电",
    "石油石化",
    "综合",
    "银行",
]

# 字段名
DATE_COL = "date"
INSTRUMENT_COL = "instrument"
FACTOR_COL = "exp_wgt_return_6m"
RET_COL = "fwd_ret"
INDUSTRY_COL = "cs_level1_name"
MKT_CAP_COL = "total_market_cap"

# 未来收益口径
# close_to_close：
#   t日因子，对应 t+REBALANCE_DAYS 收盘 / t日收盘 - 1
#
# next_open_to_next_rebalance_open：
#   t日因子，t+1开盘买入，持有 REBALANCE_DAYS 个交易日，
#   对应 t+REBALANCE_DAYS+1 开盘 / t+1 开盘 - 1
#
# 你的 BigTrader 回测是 T日信号、下一根K线开盘成交，
# 所以更贴近回测的是 next_open_to_next_rebalance_open
RETURN_MODE = "next_open_to_next_rebalance_open"


# =====================================================
# 2. 构造 exp_wgt_return_6m 因子表达式
# =====================================================

def lag(field: str, k: int) -> str:
    """生成 BigQuant DAI SQL 的 m_lag 表达式。k=0 表示当前值。"""
    if k == 0:
        return field
    return f"m_lag({field}, {k})"


num_terms = []
den_terms = []

for i in range(LOOKBACK_DAYS):
    # 指数衰减权重：exp(-x_i / N / 4)
    # N = FACTOR_MONTHS = 6
    decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)

    close_i = lag("close", i)
    close_i_1 = lag("close", i + 1)
    turn_i = lag("turn", i)

    # 第 t-i 日收益率
    ret_i = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

    # 分子：收益率 * 换手率 * 指数衰减权重
    num_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i} * {ret_i}), 0.0)"
    )

    # 分母：换手率 * 指数衰减权重
    den_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)"
    )


num_expr = " + ".join(num_terms)
den_expr = " + ".join(den_terms)

industry_sql = ", ".join([f"'{x}'" for x in EXCLUDE_INDUSTRIES])

calc_start_date = (
    datetime.strptime(START_DATE, "%Y-%m-%d") - timedelta(days=BEFORE_START_DAYS)
).strftime("%Y-%m-%d")

calc_end_date = (
    datetime.strptime(END_DATE, "%Y-%m-%d") + timedelta(days=AFTER_END_DAYS)
).strftime("%Y-%m-%d")


# =====================================================
# 3. 构造未来收益率表达式
# =====================================================

if RETURN_MODE == "close_to_close":
    fwd_ret_expr = f"""
        m_lead(close, {FORWARD_DAYS}) / NULLIF(close, 0) - 1.0
    """
elif RETURN_MODE == "next_open_to_next_rebalance_open":
    fwd_ret_expr = f"""
        m_lead(open, {FORWARD_DAYS + 1}) / NULLIF(m_lead(open, 1), 0) - 1.0
    """
else:
    raise ValueError("RETURN_MODE 只能是 close_to_close 或 next_open_to_next_rebalance_open")


# =====================================================
# 4. 用 DAI SQL 拉取完整候选池数据
# =====================================================

if USE_STRATEGY_UNIVERSE:
    universe_filter_sql = f"""
      AND mcap_pct <= 1.0 / 3.0
      AND cs_level1_name NOT IN ({industry_sql})
    """
else:
    universe_filter_sql = ""


sql = f"""
WITH factor_raw AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        list_days,
        is_risk_warning,
        suspended,

        close,
        open,
        turn,

        -- 市值从小到大做截面百分位排名
        c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,

        -- 华泰 exp_wgt_return_6m 因子
        ({num_expr}) / NULLIF(({den_expr}), 0) AS exp_wgt_return_6m,

        -- 未来收益率，周期绑定 REBALANCE_DAYS
        {fwd_ret_expr} AS fwd_ret

    FROM cn_stock_prefactors

    WHERE date >= '{calc_start_date}'
      AND date <= '{calc_end_date}'

      -- 只取沪深A股，剔除北交所
      AND list_sector IN (1, 2, 3)

      -- 剔除ST / 风险警示
      AND is_risk_warning = 0

      -- 剔除停牌
      AND suspended = 0

      -- 剔除上市时间过短的股票
      AND list_days >= {MIN_LIST_DAYS}

      -- 基础字段非空
      AND close IS NOT NULL
      AND open IS NOT NULL
      AND turn IS NOT NULL
      AND total_market_cap IS NOT NULL
      AND cs_level1_name IS NOT NULL
),

universe AS (
    SELECT
        *
    FROM factor_raw
    WHERE date >= '{START_DATE}'
      AND date <= '{END_DATE}'
      AND exp_wgt_return_6m IS NOT NULL
      AND fwd_ret IS NOT NULL
      AND total_market_cap > 0

      {universe_filter_sql}
)

SELECT
    date,
    instrument,
    cs_level1_name,
    total_market_cap,
    mcap_pct,
    exp_wgt_return_6m,
    fwd_ret

FROM universe

ORDER BY date ASC, instrument ASC
"""

df = dai.query(sql).df()

df[DATE_COL] = pd.to_datetime(df[DATE_COL])

if df.empty:
    raise ValueError("df 为空，请检查日期范围、字段名称、行业过滤或市值过滤条件。")


# =====================================================
# 5. 生成调仓截面
# =====================================================

all_dates = sorted(df[DATE_COL].dropna().unique())

# 每隔 REBALANCE_DAYS 个交易日取一个截面
rebalance_dates = all_dates[::REBALANCE_DAYS]
rebalance_dates_set = set(rebalance_dates)

df_rebalance = df[df[DATE_COL].isin(rebalance_dates_set)].copy()

if df_rebalance.empty:
    raise ValueError("调仓截面为空，请检查 REBALANCE_DAYS 或样本区间。")


# =====================================================
# 6. 截面预处理：去极值、标准化、市值取对数
# =====================================================

data = df_rebalance.copy()

data = data.replace([np.inf, -np.inf], np.nan)

needed_cols = [
    DATE_COL,
    INSTRUMENT_COL,
    FACTOR_COL,
    RET_COL,
    INDUSTRY_COL,
    MKT_CAP_COL,
]

data = data.dropna(subset=needed_cols).copy()
data = data[data[MKT_CAP_COL] > 0].copy()

data["ln_mktcap"] = np.log(data[MKT_CAP_COL])


def winsorize_series(s, q_low=0.01, q_high=0.99):
    low = s.quantile(q_low)
    high = s.quantile(q_high)
    return s.clip(lower=low, upper=high)


def zscore_series(s):
    std = s.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(np.nan, index=s.index)
    return (s - s.mean()) / std


def preprocess_by_date(g):
    """
    每个调仓日截面内：
    1. 因子去极值
    2. 因子标准化
    3. 市值去极值
    4. 市值标准化
    """
    g = g.copy()

    g["factor_winsor"] = winsorize_series(
        g[FACTOR_COL],
        WINSOR_Q_LOW,
        WINSOR_Q_HIGH
    )

    g["factor_z"] = zscore_series(g["factor_winsor"])

    g["ln_mktcap_winsor"] = winsorize_series(
        g["ln_mktcap"],
        WINSOR_Q_LOW,
        WINSOR_Q_HIGH
    )

    g["ln_mktcap_z"] = zscore_series(g["ln_mktcap_winsor"])

    return g


data = data.groupby(DATE_COL, group_keys=False).apply(preprocess_by_date)

data = data.dropna(
    subset=[
        "factor_z",
        RET_COL,
        INDUSTRY_COL,
        "ln_mktcap_z",
    ]
).copy()

if data.empty:
    raise ValueError("预处理后 data 为空，请检查因子值是否大量为常数或缺失。")


# =====================================================
# 7. 中性化与统计工具函数
# =====================================================

def build_controls(g):
    """
    构造控制变量：
    1. ln_mktcap_z：标准化后的对数市值
    2. 行业虚拟变量：cs_level1_name dummies
    """
    industry_dummies = pd.get_dummies(
        g[INDUSTRY_COL].astype(str),
        prefix="industry",
        drop_first=True
    )

    X = pd.concat(
        [
            g[["ln_mktcap_z"]].astype(float),
            industry_dummies.astype(float),
        ],
        axis=1
    )

    X = sm.add_constant(X, has_constant="add")
    return X


def neutralize_series(y, X):
    """
    对 y 做行业 + 市值中性化，返回残差。
    """
    y = y.astype(float)

    try:
        model = sm.OLS(y, X, missing="drop").fit()
        resid = model.resid
        return resid.reindex(y.index)
    except Exception:
        return pd.Series(np.nan, index=y.index)


def calc_corr(x, y, method="pearson"):
    """
    计算相关系数。
    method:
    - pearson：普通 IC
    - spearman：Rank IC
    """
    tmp = pd.concat([x, y], axis=1).dropna()

    if len(tmp) < MIN_OBS:
        return np.nan

    return tmp.iloc[:, 0].corr(tmp.iloc[:, 1], method=method)


# =====================================================
# 8. 每个调仓截面计算 IC 和回归法因子收益率
# =====================================================

period_results = []

for date, g in data.groupby(DATE_COL):
    g = g.dropna(
        subset=[
            "factor_z",
            RET_COL,
            INDUSTRY_COL,
            "ln_mktcap_z",
        ]
    ).copy()

    if len(g) < MIN_OBS:
        continue

    y = g[RET_COL].astype(float)

    # -------------------------------------------------
    # A. 原始 IC / Rank IC
    # -------------------------------------------------

    raw_ic = calc_corr(
        g["factor_z"],
        y,
        method="pearson"
    )

    raw_rank_ic = calc_corr(
        g["factor_z"],
        y,
        method="spearman"
    )

    # -------------------------------------------------
    # B. 中性化 IC / Rank IC
    #
    # 1. 因子对 行业 + 市值 回归，取残差
    # 2. 未来收益率对 行业 + 市值 回归，取残差
    # 3. 两个残差再计算相关性
    # -------------------------------------------------

    X_controls = build_controls(g)

    factor_neutral = neutralize_series(
        g["factor_z"],
        X_controls
    )

    ret_neutral = neutralize_series(
        y,
        X_controls
    )

    neutral_ic = calc_corr(
        factor_neutral,
        ret_neutral,
        method="pearson"
    )

    neutral_rank_ic = calc_corr(
        factor_neutral,
        ret_neutral,
        method="spearman"
    )

    # -------------------------------------------------
    # C. 原始回归法因子收益率
    #
    # fwd_ret = alpha + beta * factor_z + error
    # beta = 原始回归法因子收益率
    # -------------------------------------------------

    try:
        X_raw = sm.add_constant(
            g[["factor_z"]].astype(float),
            has_constant="add"
        )

        raw_model = sm.OLS(y, X_raw, missing="drop").fit()

        raw_factor_return = raw_model.params.get("factor_z", np.nan)

    except Exception:
        raw_factor_return = np.nan

    # -------------------------------------------------
    # D. 中性化回归法因子收益率
    #
    # fwd_ret = alpha
    #         + beta * factor_z
    #         + gamma * ln_mktcap_z
    #         + industry_dummies
    #         + error
    #
    # beta = 行业、市值中性化后的回归法因子收益率
    # -------------------------------------------------

    try:
        X_neutral = pd.concat(
            [
                g[["factor_z"]].astype(float),
                X_controls.drop(columns=["const"], errors="ignore"),
            ],
            axis=1
        )

        X_neutral = sm.add_constant(
            X_neutral,
            has_constant="add"
        )

        neutral_model = sm.OLS(y, X_neutral, missing="drop").fit()

        neutral_factor_return = neutral_model.params.get("factor_z", np.nan)

    except Exception:
        neutral_factor_return = np.nan

    period_results.append({
        "date": date,
        "n_stocks": len(g),

        "raw_ic": raw_ic,
        "raw_rank_ic": raw_rank_ic,
        "neutral_ic": neutral_ic,
        "neutral_rank_ic": neutral_rank_ic,

        "raw_factor_return": raw_factor_return,
        "neutral_factor_return": neutral_factor_return,
    })


result = pd.DataFrame(period_results)

if result.empty:
    raise ValueError("result 为空，可能是调仓截面样本数不足，建议降低 MIN_OBS 或放宽股票池条件。")

result = result.sort_values("date").reset_index(drop=True)


# =====================================================
# 9. 汇总统计：ICIR 和回归因子收益率 IR
# =====================================================

def calc_mean_std_ir_direction(s):
    """
    返回：
    1. 均值
    2. IR / ICIR = 均值 / 标准差
    3. 方向正确率

    对 IC / Rank IC：
        IR / ICIR = IC均值 / IC标准差

    对回归因子收益率：
        IR = 因子收益率均值 / 因子收益率标准差
    """
    s = s.dropna()

    if len(s) == 0:
        return np.nan, np.nan, np.nan

    mean = s.mean()
    std = s.std(ddof=1)

    ir = mean / std if std != 0 else np.nan

    # FACTOR_DIRECTION = -1 时，s < 0 才算方向正确
    # FACTOR_DIRECTION = 1 时，s > 0 才算方向正确
    direction_accuracy = (FACTOR_DIRECTION * s > 0).mean()

    return mean, ir, direction_accuracy


raw_rank_ic_mean, raw_rank_icir, raw_rank_ic_direction = calc_mean_std_ir_direction(
    result["raw_rank_ic"]
)

neutral_rank_ic_mean, neutral_rank_icir, neutral_rank_ic_direction = calc_mean_std_ir_direction(
    result["neutral_rank_ic"]
)

raw_factor_return_mean, raw_factor_return_ir, raw_factor_return_direction = calc_mean_std_ir_direction(
    result["raw_factor_return"]
)

neutral_factor_return_mean, neutral_factor_return_ir, neutral_factor_return_direction = calc_mean_std_ir_direction(
    result["neutral_factor_return"]
)


# =====================================================
# 10. 最终只展示核心结论表
# =====================================================

key_conclusion = pd.DataFrame([
    [
        "原始RankIC",
        raw_rank_ic_mean,
        raw_rank_icir,
        raw_rank_ic_direction,
    ],
    [
        "中性化RankIC",
        neutral_rank_ic_mean,
        neutral_rank_icir,
        neutral_rank_ic_direction,
    ],
    [
        "原始回归因子收益率",
        raw_factor_return_mean,
        raw_factor_return_ir,
        raw_factor_return_direction,
    ],
    [
        "中性化回归因子收益率",
        neutral_factor_return_mean,
        neutral_factor_return_ir,
        neutral_factor_return_direction,
    ],
], columns=["核心指标", "均值", "IR / ICIR", "方向正确率"])

key_conclusion[["均值", "IR / ICIR", "方向正确率"]] = key_conclusion[
    ["均值", "IR / ICIR", "方向正确率"]
].round(6)

display(key_conclusion)

由于根据RankIC来看因子的排序选股能力是较强的，但策略表现效果却没有很好，于是考虑先对因子单调性使用分组回测方法来检查

In [ ]:
import math
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import dai
import matplotlib.pyplot as plt
from IPython.display import display


# =====================================================
# 1. 参数区
# =====================================================

START_DATE = "2024-01-01"
END_DATE = "2026-06-01"

# 持仓 / 调仓周期
REBALANCE_DAYS = 20
FORWARD_DAYS = REBALANCE_DAYS

# 分组数量
GROUP_NUM = 10

# exp_wgt_return_6m：6个月动量因子
FACTOR_MONTHS = 6
LOOKBACK_DAYS = 21 * FACTOR_MONTHS

# 为了计算 m_lag，需要在开始日前多取历史
BEFORE_START_DAYS = 300

# 为了计算未来收益，需要在结束日后多取数据
AFTER_END_DAYS = int(FORWARD_DAYS * 365 / 252) + 30

# list_days 是自然日口径
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 是否沿用原策略里的行业剔除
USE_INDUSTRY_EXCLUDE = True

# 原策略剔除行业
EXCLUDE_INDUSTRIES = [
    "交通运输",
    "电力及公用事业",
    "纺织服装",
    "轻工制造",
    "家电",
    "石油石化",
    "综合",
    "银行",
]

# 分组收益是否使用更贴近回测的 T+1 开盘买入、T+N+1 开盘卖出口径
# True：使用 T+1 open -> T+N+1 open
# False：使用 T close -> T+N close
USE_NEXT_OPEN_RETURN = True

# 是否在分组回测里加入简单可交易过滤
# 建议先用 False 看纯因子单调性
APPLY_TRADABILITY_FILTER = False

# 字段名
DATE_COL = "date"
INSTRUMENT_COL = "instrument"
FACTOR_COL = "exp_wgt_return_6m"
RET_COL = "fwd_ret"
INDUSTRY_COL = "cs_level1_name"
MKT_CAP_COL = "total_market_cap"


# =====================================================
# 2. 构造 exp_wgt_return_6m 因子表达式
# =====================================================

def lag(field: str, k: int) -> str:
    if k == 0:
        return field
    return f"m_lag({field}, {k})"


num_terms = []
den_terms = []

for i in range(LOOKBACK_DAYS):
    decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)

    close_i = lag("close", i)
    close_i_1 = lag("close", i + 1)
    turn_i = lag("turn", i)

    ret_i = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

    num_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i} * {ret_i}), 0.0)"
    )

    den_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)"
    )

num_expr = " + ".join(num_terms)
den_expr = " + ".join(den_terms)

industry_sql = ", ".join([f"'{x}'" for x in EXCLUDE_INDUSTRIES])

calc_start_date = (
    datetime.strptime(START_DATE, "%Y-%m-%d") - timedelta(days=BEFORE_START_DAYS)
).strftime("%Y-%m-%d")

calc_end_date = (
    datetime.strptime(END_DATE, "%Y-%m-%d") + timedelta(days=AFTER_END_DAYS)
).strftime("%Y-%m-%d")


# =====================================================
# 3. 构造未来收益率表达式
# =====================================================

if USE_NEXT_OPEN_RETURN:
    # T日因子，T+1开盘买入，T+FORWARD_DAYS+1开盘卖出
    fwd_ret_expr = f"""
        m_lead(open, {FORWARD_DAYS + 1}) / NULLIF(m_lead(open, 1), 0) - 1.0
    """
else:
    # T日因子，T日收盘到T+FORWARD_DAYS日收盘
    fwd_ret_expr = f"""
        m_lead(close, {FORWARD_DAYS}) / NULLIF(close, 0) - 1.0
    """


# =====================================================
# 4. 股票池过滤条件
# =====================================================

if USE_INDUSTRY_EXCLUDE:
    industry_filter_sql = f"""
      AND cs_level1_name NOT IN ({industry_sql})
    """
else:
    industry_filter_sql = ""


if APPLY_TRADABILITY_FILTER:
    tradability_filter_sql = f"""
      AND next_suspended = 0
      AND exit_suspended = 0
      AND next_open < next_upper_limit
      AND exit_open > exit_lower_limit
    """
else:
    tradability_filter_sql = ""


# =====================================================
# 5. 用 DAI SQL 拉取小市值股票池数据
# =====================================================

sql = f"""
WITH factor_raw AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        list_days,
        is_risk_warning,
        suspended,

        close,
        open,
        turn,

        upper_limit,
        lower_limit,

        -- 下一交易日买入时点
        m_lead(open, 1) AS next_open,
        m_lead(upper_limit, 1) AS next_upper_limit,
        m_lead(lower_limit, 1) AS next_lower_limit,
        m_lead(suspended, 1) AS next_suspended,

        -- 退出时点
        m_lead(open, {FORWARD_DAYS + 1}) AS exit_open,
        m_lead(upper_limit, {FORWARD_DAYS + 1}) AS exit_upper_limit,
        m_lead(lower_limit, {FORWARD_DAYS + 1}) AS exit_lower_limit,
        m_lead(suspended, {FORWARD_DAYS + 1}) AS exit_suspended,

        -- 市值从小到大做截面百分位排名
        c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,

        -- 华泰 exp_wgt_return_6m 因子
        ({num_expr}) / NULLIF(({den_expr}), 0) AS exp_wgt_return_6m,

        -- 未来持仓期收益
        {fwd_ret_expr} AS fwd_ret

    FROM cn_stock_prefactors

    WHERE date >= '{calc_start_date}'
      AND date <= '{calc_end_date}'

      -- 只取沪深A股，剔除北交所
      AND list_sector IN (1, 2, 3)

      -- 剔除ST / 风险警示
      AND is_risk_warning = 0

      -- 剔除停牌
      AND suspended = 0

      -- 剔除上市时间过短
      AND list_days >= {MIN_LIST_DAYS}

      -- 基础字段非空
      AND close IS NOT NULL
      AND open IS NOT NULL
      AND turn IS NOT NULL
      AND total_market_cap IS NOT NULL
      AND cs_level1_name IS NOT NULL
),

universe AS (
    SELECT
        *
    FROM factor_raw
    WHERE date >= '{START_DATE}'
      AND date <= '{END_DATE}'

      -- 小市值股票池：市值排名后1/3
      AND mcap_pct <= 1.0 / 3.0

      -- 是否沿用原策略行业剔除
      {industry_filter_sql}

      -- 因子和未来收益非空
      AND exp_wgt_return_6m IS NOT NULL
      AND fwd_ret IS NOT NULL
      AND total_market_cap > 0

      -- 可选：简单交易可行性过滤
      {tradability_filter_sql}
)

SELECT
    date,
    instrument,
    cs_level1_name,
    total_market_cap,
    mcap_pct,
    exp_wgt_return_6m,
    fwd_ret

FROM universe

ORDER BY date ASC, instrument ASC
"""

df = dai.query(sql).df()

df[DATE_COL] = pd.to_datetime(df[DATE_COL])

if df.empty:
    raise ValueError("df 为空，请检查日期范围、字段名称、行业过滤或小市值过滤条件。")


# =====================================================
# 6. 生成调仓截面
# =====================================================

all_dates = sorted(df[DATE_COL].dropna().unique())

rebalance_dates = all_dates[::REBALANCE_DAYS]
rebalance_dates_set = set(rebalance_dates)

df_rebalance = df[df[DATE_COL].isin(rebalance_dates_set)].copy()

if df_rebalance.empty:
    raise ValueError("df_rebalance 为空，请检查 REBALANCE_DAYS 或样本区间。")


# =====================================================
# 7. 在每个调仓截面内按因子分组
# =====================================================

def assign_factor_groups(g):
    """
    每个调仓日截面内：
    按 exp_wgt_return_6m 从低到高分组。

    G1：因子值最低组
    G10：因子值最高组

    因为你的策略是买因子值最低的股票，
    所以理想状态是 G1 收益最高，G10 收益最低。
    """
    g = g.copy()

    if len(g) < GROUP_NUM:
        g["group"] = np.nan
        return g

    # 使用 rank 后再 qcut，避免因子大量重复值导致 qcut 报错
    g["_factor_rank"] = g[FACTOR_COL].rank(
        method="first",
        ascending=True
    )

    g["group"] = pd.qcut(
        g["_factor_rank"],
        q=GROUP_NUM,
        labels=[f"G{i}" for i in range(1, GROUP_NUM + 1)]
    )

    return g


grouped_df = df_rebalance.groupby(
    DATE_COL,
    group_keys=False
).apply(assign_factor_groups)

grouped_df = grouped_df.dropna(subset=["group"]).copy()

grouped_df["group"] = grouped_df["group"].astype(str)


# =====================================================
# 8. 计算每期各组等权收益
# =====================================================

period_group_return = grouped_df.groupby(
    [DATE_COL, "group"]
).agg(
    period_ret=(RET_COL, "mean"),
    n_stocks=(INSTRUMENT_COL, "nunique"),
    factor_mean=(FACTOR_COL, "mean"),
    factor_median=(FACTOR_COL, "median"),
    mcap_mean=(MKT_CAP_COL, "mean"),
).reset_index()

group_ret_pivot = period_group_return.pivot(
    index=DATE_COL,
    columns="group",
    values="period_ret"
).sort_index()

# 保证列顺序 G1, G2, ..., G10
group_cols = [f"G{i}" for i in range(1, GROUP_NUM + 1)]
group_ret_pivot = group_ret_pivot[group_cols]

group_cum_ret = (1.0 + group_ret_pivot.fillna(0.0)).cumprod() - 1.0


# =====================================================
# 9. 分组绩效统计函数
# =====================================================

def calc_max_drawdown(period_ret):
    wealth = (1.0 + period_ret.fillna(0.0)).cumprod()
    drawdown = wealth / wealth.cummax() - 1.0
    return drawdown.min()


def calc_group_metrics(period_ret):
    """
    period_ret 是每个调仓周期的收益率序列。
    """
    r = period_ret.dropna()

    if len(r) == 0:
        return pd.Series({
            "样本期数": 0,
            "周期收益均值": np.nan,
            "累计收益率": np.nan,
            "年化收益率": np.nan,
            "年化波动率": np.nan,
            "收益IR": np.nan,
            "最大回撤": np.nan,
            "胜率": np.nan,
        })

    annual_factor = 252 / REBALANCE_DAYS

    wealth = (1.0 + r).cumprod()
    total_ret = wealth.iloc[-1] - 1.0

    annual_ret = wealth.iloc[-1] ** (annual_factor / len(r)) - 1.0

    annual_vol = r.std(ddof=1) * np.sqrt(annual_factor)

    ret_ir = (
        r.mean() / r.std(ddof=1) * np.sqrt(annual_factor)
        if r.std(ddof=1) != 0
        else np.nan
    )

    max_dd = calc_max_drawdown(r)

    win_rate = (r > 0).mean()

    return pd.Series({
        "样本期数": len(r),
        "周期收益均值": r.mean(),
        "累计收益率": total_ret,
        "年化收益率": annual_ret,
        "年化波动率": annual_vol,
        "收益IR": ret_ir,
        "最大回撤": max_dd,
        "胜率": win_rate,
    })


group_metrics = pd.DataFrame({
    group: calc_group_metrics(group_ret_pivot[group])
    for group in group_cols
}).T

# 补充分组的平均因子值和平均市值
group_factor_mcap = period_group_return.groupby("group").agg(
    因子均值=( "factor_mean", "mean" ),
    因子中位数=( "factor_median", "mean" ),
    平均市值=( "mcap_mean", "mean" ),
)

group_metrics = group_metrics.join(group_factor_mcap)

# 调整列顺序
group_metrics = group_metrics[
    [
        "样本期数",
        "因子均值",
        "因子中位数",
        "平均市值",
        "周期收益均值",
        "累计收益率",
        "年化收益率",
        "年化波动率",
        "收益IR",
        "最大回撤",
        "胜率",
    ]
]


# =====================================================
# 10. 多空组合：G1 - G10
# =====================================================

long_short_ret = group_ret_pivot["G1"] - group_ret_pivot[f"G{GROUP_NUM}"]
long_short_metrics = calc_group_metrics(long_short_ret).to_frame("G1-G10").T


# =====================================================
# 11. 单调性检验
# =====================================================

group_number = pd.Series(
    range(1, GROUP_NUM + 1),
    index=group_cols
)

annual_ret_by_group = group_metrics["年化收益率"]
mean_ret_by_group = group_metrics["周期收益均值"]

# 因为 G1 是低因子值，G10 是高因子值；
# 如果低因子值更好，则组号越大，收益越低，Spearman 应该为负。
spearman_annual = group_number.corr(
    annual_ret_by_group,
    method="spearman"
)

spearman_mean = group_number.corr(
    mean_ret_by_group,
    method="spearman"
)

g1_annual = group_metrics.loc["G1", "年化收益率"]
g_last_annual = group_metrics.loc[f"G{GROUP_NUM}", "年化收益率"]

g1_total = group_metrics.loc["G1", "累计收益率"]
g_last_total = group_metrics.loc[f"G{GROUP_NUM}", "累计收益率"]

monotonic_summary = pd.DataFrame([
    [
        "组号与年化收益率Spearman相关",
        spearman_annual,
        "越接近 -1，说明低因子组越强、高因子组越弱，单调性越好",
    ],
    [
        "组号与周期收益均值Spearman相关",
        spearman_mean,
        "越接近 -1，说明分组周期收益越单调递减",
    ],
    [
        "G1年化收益率 - G10年化收益率",
        g1_annual - g_last_annual,
        "大于0说明最低因子组跑赢最高因子组",
    ],
    [
        "G1累计收益率 - G10累计收益率",
        g1_total - g_last_total,
        "大于0说明最低因子组长期跑赢最高因子组",
    ],
    [
        "G1-G10多空组合年化收益率",
        long_short_metrics.loc["G1-G10", "年化收益率"],
        "越高说明多低因子、空高因子的收益越强",
    ],
    [
        "G1-G10多空组合最大回撤",
        long_short_metrics.loc["G1-G10", "最大回撤"],
        "绝对值越小越好",
    ],
], columns=["检验项", "数值", "解释"])


# =====================================================
# 12. 展示结果
# =====================================================

display_group_metrics = group_metrics.copy()

pct_cols = [
    "周期收益均值",
    "累计收益率",
    "年化收益率",
    "年化波动率",
    "最大回撤",
    "胜率",
]

for col in pct_cols:
    display_group_metrics[col] = display_group_metrics[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else ""
    )

display_group_metrics["收益IR"] = display_group_metrics["收益IR"].map(
    lambda x: f"{x:.3f}" if pd.notna(x) else ""
)

display_group_metrics["因子均值"] = display_group_metrics["因子均值"].map(
    lambda x: f"{x:.6f}" if pd.notna(x) else ""
)

display_group_metrics["因子中位数"] = display_group_metrics["因子中位数"].map(
    lambda x: f"{x:.6f}" if pd.notna(x) else ""
)

display_group_metrics["平均市值"] = display_group_metrics["平均市值"].map(
    lambda x: f"{x / 1e8:.2f}亿" if pd.notna(x) else ""
)

print("一、分组绩效表：G1为因子值最低组，G10为因子值最高组")
display(display_group_metrics)


display_mono = monotonic_summary.copy()

display_mono["数值"] = display_mono["数值"].map(
    lambda x: f"{x:.6f}" if pd.notna(x) else ""
)

print("二、单调性检验表")
display(display_mono)


display_ls = long_short_metrics.copy()

for col in [
    "周期收益均值",
    "累计收益率",
    "年化收益率",
    "年化波动率",
    "最大回撤",
    "胜率",
]:
    display_ls[col] = display_ls[col].map(
        lambda x: f"{x:.2%}" if pd.notna(x) else ""
    )

display_ls["收益IR"] = display_ls["收益IR"].map(
    lambda x: f"{x:.3f}" if pd.notna(x) else ""
)

print("三、G1-G10 多空组合表现")
display(display_ls)


# =====================================================
# 13. 画图：分组累计收益曲线
# =====================================================

plt.figure(figsize=(13, 6))

for group in group_cols:
    plt.plot(
        group_cum_ret.index,
        group_cum_ret[group],
        label=group
    )

plt.title(
    f"小市值股票池内 exp_wgt_return_6m 分组累计收益｜"
    f"G1最低因子值，G{GROUP_NUM}最高因子值"
)
plt.xlabel("日期")
plt.ylabel("累计收益率")
plt.legend(ncol=5)
plt.grid(True)
plt.show()


# =====================================================
# 14. 画图：各组年化收益率柱状图
# =====================================================

plt.figure(figsize=(10, 5))

group_metrics["年化收益率"].plot(kind="bar")

plt.title("各分组年化收益率")
plt.xlabel("分组")
plt.ylabel("年化收益率")
plt.grid(True, axis="y")
plt.show()


# =====================================================
# 15. 画图：G1-G10多空累计收益
# =====================================================

long_short_cum_ret = (1.0 + long_short_ret.fillna(0.0)).cumprod() - 1.0

plt.figure(figsize=(12, 5))

plt.plot(
    long_short_cum_ret.index,
    long_short_cum_ret,
    label=f"G1 - G{GROUP_NUM}"
)

plt.title(f"G1-G{GROUP_NUM} 多空组合累计收益")
plt.xlabel("日期")
plt.ylabel("累计收益率")
plt.legend()
plt.grid(True)
plt.show()

从当前的分组回测效果上来看，该因子在头部分组的单调性是良好的，因此策略曲线的表现很可能是因为被大幅回撤所拖累，考虑在策略当中加入防御性策略，这里采用指数趋势过滤逻辑，同时为了保证结果的严谨性，还加入涨跌停交易限制，确保回测结果不会过于乐观，以中证1000指数的20日均线作为分界，如果跌破则降低仓位，如果站上则调高仓位

In [ ]:
import math
import hashlib
import ast
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

import dai
from bigquant import bigtrader
from IPython.display import display


# =====================================================
# 1. 策略参数区
# =====================================================

START_DATE = "2020-01-01"
END_DATE = "2026-06-01"

# 每次持有股票数量
HOLD_NUM = 100

# 每隔多少个交易日重新选股一次
REBALANCE_DAYS = 20

# 强势市场目标总仓位
TARGET_TOTAL_WEIGHT = 0.98

# 弱势市场防御总仓位
# 如果想弱势时完全空仓，改成 0.0
DEFENSIVE_TOTAL_WEIGHT = 0.30

# 趋势过滤指数
# 小市值策略建议用中证1000
TREND_INDEX = "000852.SH"

# 指数趋势均线天数
TREND_MA_DAYS = 30

# exp_wgt_return_6m：6个月动量因子
FACTOR_MONTHS = 6
LOOKBACK_DAYS = 21 * FACTOR_MONTHS

# 为了计算 m_lag 和指数均线，需要在回测开始日前多取一段历史
BEFORE_START_DAYS = max(300, TREND_MA_DAYS * 3)

# 为了计算下一交易日涨跌停，需要在结束日后多取一段数据
AFTER_END_DAYS = 30

# list_days 是自然日口径，不是交易日口径
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 初始资金
CAPITAL_BASE = 1_000_000

# 剔除行业：默认使用中信一级行业 cs_level1_name
EXCLUDE_INDUSTRIES = [
    "交通运输",
    "电力及公用事业",
    "纺织服装",
    "轻工制造",
    "家电",
    "石油石化",
    "综合",
    "银行",
]

# 回测基准
BENCHMARK = "000300.SH"

# 是否打印每日调仓 / 风控日志
VERBOSE = False

# 手续费参数，需要和 context.set_commission 保持一致
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COST = 5.0


# =====================================================
# 2. 构造 exp_wgt_return_6m 因子表达式
# =====================================================

def lag(field: str, k: int) -> str:
    """生成 BigQuant DAI SQL 的 m_lag 表达式。k=0 表示当前值。"""
    if k == 0:
        return field
    return f"m_lag({field}, {k})"


num_terms = []
den_terms = []

for i in range(LOOKBACK_DAYS):
    decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)

    close_i = lag("close", i)
    close_i_1 = lag("close", i + 1)
    turn_i = lag("turn", i)

    ret_i = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

    num_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i} * {ret_i}), 0.0)"
    )

    den_terms.append(
        f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)"
    )


num_expr = " + ".join(num_terms)
den_expr = " + ".join(den_terms)

industry_sql = ", ".join([f"'{x}'" for x in EXCLUDE_INDUSTRIES])

calc_start_date = (
    datetime.strptime(START_DATE, "%Y-%m-%d") - timedelta(days=BEFORE_START_DAYS)
).strftime("%Y-%m-%d")

calc_end_date = (
    datetime.strptime(END_DATE, "%Y-%m-%d") + timedelta(days=AFTER_END_DAYS)
).strftime("%Y-%m-%d")


# =====================================================
# 3. 查询：每日候选目标股票 + 每日指数趋势信号
# =====================================================

signal_sql = f"""
WITH index_raw AS (
    SELECT
        date,
        instrument,
        close AS trend_index_close,

        AVG(close) OVER (
            PARTITION BY instrument
            ORDER BY date
            ROWS BETWEEN {TREND_MA_DAYS - 1} PRECEDING AND CURRENT ROW
        ) AS trend_index_ma

    FROM cn_stock_index_bar1d

    WHERE instrument = '{TREND_INDEX}'
      AND date >= '{calc_start_date}'
      AND date <= '{END_DATE}'
),

index_trend AS (
    SELECT
        date,
        trend_index_close,
        trend_index_ma,

        CASE
            WHEN trend_index_ma IS NULL THEN 0
            WHEN trend_index_close > trend_index_ma THEN 1
            ELSE 0
        END AS risk_on,

        CASE
            WHEN trend_index_ma IS NULL THEN {DEFENSIVE_TOTAL_WEIGHT}
            WHEN trend_index_close > trend_index_ma THEN {TARGET_TOTAL_WEIGHT}
            ELSE {DEFENSIVE_TOTAL_WEIGHT}
        END AS trend_total_weight

    FROM index_raw
),

factor_raw AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        list_days,
        is_risk_warning,
        suspended,

        close,
        turn,

        c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,

        ({num_expr}) / NULLIF(({den_expr}), 0) AS exp_wgt_return_6m

    FROM cn_stock_prefactors

    WHERE date >= '{calc_start_date}'
      AND date <= '{END_DATE}'

      -- 只取沪深A股，剔除北交所
      AND list_sector IN (1, 2, 3)

      -- 剔除ST / 风险警示
      AND is_risk_warning = 0

      -- 剔除停牌
      AND suspended = 0

      -- 剔除上市时间过短
      AND list_days >= {MIN_LIST_DAYS}

      AND close IS NOT NULL
      AND turn IS NOT NULL
      AND total_market_cap IS NOT NULL
      AND cs_level1_name IS NOT NULL
),

universe AS (
    SELECT
        fr.*,
        it.trend_index_close,
        it.trend_index_ma,
        it.risk_on,
        it.trend_total_weight

    FROM factor_raw fr

    LEFT JOIN index_trend it
        ON fr.date = it.date

    WHERE fr.date >= '{START_DATE}'

      -- 小市值股票池：市值排名后1/3
      AND fr.mcap_pct <= 1.0 / 3.0

      -- 剔除指定中信一级行业
      AND fr.cs_level1_name NOT IN ({industry_sql})

      -- 因子值非空
      AND fr.exp_wgt_return_6m IS NOT NULL

      -- 指数趋势信号非空
      AND it.trend_total_weight IS NOT NULL
),

ranked AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        exp_wgt_return_6m,

        trend_index_close,
        trend_index_ma,
        risk_on,
        trend_total_weight,

        row_number() OVER (
            PARTITION BY date
            ORDER BY exp_wgt_return_6m ASC, total_market_cap ASC, instrument ASC
        ) AS factor_rank

    FROM universe
)

SELECT
    date,
    instrument,
    cs_level1_name,
    total_market_cap,
    exp_wgt_return_6m,
    factor_rank,

    trend_index_close,
    trend_index_ma,
    risk_on,
    trend_total_weight

FROM ranked

WHERE factor_rank <= {HOLD_NUM}

ORDER BY date ASC, factor_rank ASC, instrument ASC
"""

target_daily_df = dai.query(signal_sql).df()
target_daily_df["date"] = pd.to_datetime(target_daily_df["date"]).dt.strftime("%Y-%m-%d")

if target_daily_df.empty:
    raise ValueError("target_daily_df 为空，请检查日期范围、指数代码、行业过滤、市值过滤或字段名称。")

print("每日目标信号样例：")
print(target_daily_df.head())
print("每日目标信号日期数量：", target_daily_df["date"].nunique())
print("每日目标信号股票数量：", target_daily_df["instrument"].nunique())


# =====================================================
# 4. 生成低频选股调仓日
# =====================================================

all_dates = sorted(target_daily_df["date"].unique())

rebalance_dates = all_dates[::REBALANCE_DAYS]
rebalance_dates_set = set(rebalance_dates)

target_rebalance_df = target_daily_df[
    target_daily_df["date"].isin(rebalance_dates_set)
].copy()

target_rebalance_df = target_rebalance_df.sort_values(
    ["date", "factor_rank", "instrument"]
).reset_index(drop=True)

if target_rebalance_df.empty:
    raise ValueError("target_rebalance_df 为空，请检查 REBALANCE_DAYS 或信号数据。")


# =====================================================
# 5. 构造每日持仓目标：
#    选股名单低频更新，但仓位每日跟随趋势信号调整
# =====================================================

daily_trend_df = target_daily_df.drop_duplicates("date")[
    [
        "date",
        "trend_index_close",
        "trend_index_ma",
        "risk_on",
        "trend_total_weight",
    ]
].copy()

date_to_idx = {d: i for i, d in enumerate(all_dates)}

daily_target_blocks = []

for i, rebalance_date in enumerate(rebalance_dates):
    start_idx = date_to_idx[rebalance_date]

    if i + 1 < len(rebalance_dates):
        end_idx = date_to_idx[rebalance_dates[i + 1]]
        active_dates = all_dates[start_idx:end_idx]
    else:
        active_dates = all_dates[start_idx:]

    rebalance_targets = target_rebalance_df[
        target_rebalance_df["date"] == rebalance_date
    ].copy()

    rebalance_targets = rebalance_targets[
        [
            "instrument",
            "cs_level1_name",
            "total_market_cap",
            "exp_wgt_return_6m",
            "factor_rank",
        ]
    ].copy()

    for current_date in active_dates:
        block = rebalance_targets.copy()
        block["date"] = current_date
        block["rebalance_date"] = rebalance_date
        block["is_target"] = 1
        daily_target_blocks.append(block)


daily_target_df = pd.concat(daily_target_blocks, ignore_index=True)

daily_target_df = daily_target_df.merge(
    daily_trend_df,
    on="date",
    how="left"
)

daily_target_df["stock_count"] = daily_target_df.groupby("date")["instrument"].transform("count")

daily_target_df["weight"] = (
    daily_target_df["trend_total_weight"] / daily_target_df["stock_count"]
)

daily_target_df = daily_target_df.sort_values(
    ["date", "factor_rank", "instrument"]
).reset_index(drop=True)

if daily_target_df.empty:
    raise ValueError("daily_target_df 为空，请检查每日目标构造逻辑。")


# =====================================================
# 6. 查询每日涨跌停交易状态
#    只查曾经进入过目标池的股票，降低数据量
# =====================================================

all_target_instruments = sorted(daily_target_df["instrument"].unique().tolist())
instrument_sql = ", ".join([f"'{x}'" for x in all_target_instruments])

trade_status_sql = f"""
WITH trade_raw AS (
    SELECT
        date,
        instrument,

        is_risk_warning,
        suspended,

        open,
        upper_limit,
        lower_limit,

        m_lead(open, 1) AS next_open,
        m_lead(upper_limit, 1) AS next_upper_limit,
        m_lead(lower_limit, 1) AS next_lower_limit,
        m_lead(suspended, 1) AS next_suspended

    FROM cn_stock_prefactors

    WHERE date >= '{START_DATE}'
      AND date <= '{calc_end_date}'

      AND list_sector IN (1, 2, 3)

      AND instrument IN ({instrument_sql})

      AND open IS NOT NULL
      AND upper_limit IS NOT NULL
      AND lower_limit IS NOT NULL
)

SELECT
    date,
    instrument,

    is_risk_warning,
    suspended,

    next_open,
    next_upper_limit,
    next_lower_limit,
    next_suspended,

    CASE
        WHEN next_suspended = 1 THEN 0
        WHEN next_open IS NULL THEN 0
        WHEN next_upper_limit IS NULL THEN 0
        WHEN next_open >= next_upper_limit THEN 0
        ELSE 1
    END AS can_buy_next_open,

    CASE
        WHEN next_suspended = 1 THEN 0
        WHEN next_open IS NULL THEN 0
        WHEN next_lower_limit IS NULL THEN 0
        WHEN next_open <= next_lower_limit THEN 0
        ELSE 1
    END AS can_sell_next_open

FROM trade_raw

WHERE date >= '{START_DATE}'
  AND date <= '{END_DATE}'

ORDER BY date ASC, instrument ASC
"""

trade_status_df = dai.query(trade_status_sql).df()
trade_status_df["date"] = pd.to_datetime(trade_status_df["date"]).dt.strftime("%Y-%m-%d")

if trade_status_df.empty:
    raise ValueError("trade_status_df 为空，请检查涨跌停字段或目标股票列表。")


# =====================================================
# 7. 合并每日目标仓位与每日交易状态
#    保留所有曾经入选过目标池的股票，用于非目标持仓的卖出判断
# =====================================================

signal_df = trade_status_df.merge(
    daily_target_df[
        [
            "date",
            "instrument",
            "rebalance_date",
            "cs_level1_name",
            "total_market_cap",
            "exp_wgt_return_6m",
            "factor_rank",
            "trend_index_close",
            "trend_index_ma",
            "risk_on",
            "trend_total_weight",
            "stock_count",
            "weight",
            "is_target",
        ]
    ],
    on=["date", "instrument"],
    how="left"
)

signal_df["is_target"] = signal_df["is_target"].fillna(0).astype(int)
signal_df["weight"] = signal_df["weight"].fillna(0.0)
signal_df["factor_rank"] = signal_df["factor_rank"].fillna(999999).astype(int)

# 非目标股票也补上每日趋势字段，方便统一判断
signal_df = signal_df.merge(
    daily_trend_df,
    on="date",
    how="left",
    suffixes=("", "_daily")
)

for col in ["trend_index_close", "trend_index_ma", "risk_on", "trend_total_weight"]:
    daily_col = f"{col}_daily"
    if daily_col in signal_df.columns:
        signal_df[col] = signal_df[col].combine_first(signal_df[daily_col])
        signal_df = signal_df.drop(columns=[daily_col])

signal_df = signal_df.sort_values(
    ["date", "is_target", "factor_rank", "instrument"],
    ascending=[True, False, True, True]
).reset_index(drop=True)

print("每日目标仓位样例：")
print(
    signal_df[signal_df["is_target"] == 1][
        [
            "date",
            "rebalance_date",
            "instrument",
            "factor_rank",
            "exp_wgt_return_6m",
            "risk_on",
            "trend_total_weight",
            "weight",
            "can_buy_next_open",
            "can_sell_next_open",
        ]
    ].head(30)
)

print("选股调仓次数：", len(rebalance_dates))
print("每日风控日期数量：", signal_df["date"].nunique())
print("曾经入选目标池股票数量：", len(all_target_instruments))

trend_summary = daily_trend_df.copy()
print("强势日期数量：", int((trend_summary["risk_on"] == 1).sum()))
print("防御日期数量：", int((trend_summary["risk_on"] == 0).sum()))


# =====================================================
# 8. 信号哈希
# =====================================================

check_df = daily_target_df[
    [
        "date",
        "rebalance_date",
        "instrument",
        "factor_rank",
        "risk_on",
        "trend_total_weight",
        "weight",
    ]
].copy()

check_df = check_df.sort_values(["date", "factor_rank", "instrument"])

signal_hash = hashlib.md5(
    check_df.to_csv(index=False).encode("utf-8")
).hexdigest()

print("信号哈希：", signal_hash)


# =====================================================
# 9. BigTrader 回测函数
# =====================================================

def _safe_int(x, default=0):
    try:
        if pd.isna(x):
            return default
        return int(x)
    except Exception:
        return default


def _safe_float(x, default=0.0):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def _can_buy(info):
    """
    下一交易日开盘涨停 / 停牌，则不能买。
    如果没有信息，保守起见不买。
    """
    if info is None:
        return False

    return _safe_int(info.get("can_buy_next_open", 0), 0) == 1


def _can_sell(info):
    """
    下一交易日开盘跌停 / 停牌，则不能卖。
    如果没有信息，则尝试卖出，避免持仓长期卡住。
    """
    if info is None:
        return True

    return _safe_int(info.get("can_sell_next_open", 1), 1) == 1


def initialize(context: bigtrader.IContext):
    context.signal_data = context.data.copy()

    context.signal_data["date"] = pd.to_datetime(
        context.signal_data["date"]
    ).dt.strftime("%Y-%m-%d")

    context.signal_data = context.signal_data.sort_values(
        ["date", "is_target", "factor_rank", "instrument"],
        ascending=[True, False, True, True]
    ).reset_index(drop=True)

    # 每个有趋势信号和交易状态的日期都允许进行每日仓位控制
    context.signal_dates = set(context.signal_data["date"].unique())

    # 记录上一次目标权重，用于判断这次是加仓还是减仓
    context.desired_weight_map = {}

    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COST
        )
    )


def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    today = data.current_dt.strftime("%Y-%m-%d")

    # 没有信号的日期不操作
    if today not in context.signal_dates:
        return

    today_all_df = context.signal_data[
        context.signal_data["date"] == today
    ].copy()

    if today_all_df.empty:
        return

    info_map = today_all_df.set_index("instrument").to_dict("index")

    today_target_df = today_all_df[
        today_all_df["is_target"] == 1
    ].copy()

    today_target_df = today_target_df.sort_values(
        ["factor_rank", "instrument"]
    ).reset_index(drop=True)

    target_instruments = set(today_target_df["instrument"].tolist())

    if not today_target_df.empty:
        today_total_weight = _safe_float(today_target_df["trend_total_weight"].iloc[0], 0.0)
        today_risk_on = _safe_int(today_target_df["risk_on"].iloc[0], 0)
    else:
        today_total_weight = 0.0
        today_risk_on = 0

    if VERBOSE:
        print(
            f"{today} 每日风控：risk_on={today_risk_on}, "
            f"target_total_weight={today_total_weight:.2%}"
        )

    current_positions = context.get_positions()
    holding_instruments = set(current_positions.keys())

    # =================================================
    # 1）卖出当前已经不在目标池里的股票
    #    注意：这个动作每日都会尝试。
    #    如果某只股票前一天因为跌停卖不出，后续每天继续尝试卖出。
    # =================================================

    sell_list = sorted(holding_instruments - target_instruments)

    for instrument in sell_list:
        info = info_map.get(instrument)

        if not _can_sell(info):
            if VERBOSE:
                print(f"{today} 跳过卖出 {instrument}：下一交易日开盘跌停或停牌")
            continue

        context.order_target_percent(instrument, 0)
        context.desired_weight_map[instrument] = 0.0

    # =================================================
    # 2）处理目标股票
    #    选股名单只在调仓日更新；
    #    但 weight 会每天根据指数趋势变化。
    #
    #    如果目标权重上升：需要买入，要求 can_buy。
    #    如果目标权重下降：需要卖出，要求 can_sell。
    # =================================================

    for _, row in today_target_df.iterrows():
        instrument = row["instrument"]
        target_weight = float(row["weight"])

        info = info_map.get(instrument)

        can_buy = _can_buy(info)
        can_sell = _can_sell(info)

        prev_weight = float(context.desired_weight_map.get(instrument, 0.0))
        diff = target_weight - prev_weight

        # 权重几乎不变，不重复下单
        if abs(diff) < 1e-8:
            continue

        # 目标权重为0：清仓
        if target_weight <= 0:
            if not can_sell:
                if VERBOSE:
                    print(f"{today} 跳过清仓 {instrument}：下一交易日开盘跌停或停牌")
                continue

            context.order_target_percent(instrument, 0)
            context.desired_weight_map[instrument] = 0.0
            continue

        # 需要加仓 / 新开仓
        if diff > 0:
            if not can_buy:
                if VERBOSE:
                    print(f"{today} 跳过买入/加仓 {instrument}：下一交易日开盘涨停或停牌")
                continue

            context.order_target_percent(instrument, target_weight)
            context.desired_weight_map[instrument] = target_weight
            continue

        # 需要减仓
        if diff < 0:
            if not can_sell:
                if VERBOSE:
                    print(f"{today} 跳过减仓 {instrument}：下一交易日开盘跌停或停牌")
                continue

            context.order_target_percent(instrument, target_weight)
            context.desired_weight_map[instrument] = target_weight
            continue


# =====================================================
# 10. 运行回测
# =====================================================

instruments = sorted(all_target_instruments)

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,

    start_date=START_DATE,
    end_date=END_DATE,

    capital_base=CAPITAL_BASE,
    instruments=instruments,
    data=signal_df,

    initialize=initialize,
    handle_data=handle_data,

    benchmark=BENCHMARK,

    # T日产生信号，下一根K线开盘成交
    order_price_field_buy="open",
    order_price_field_sell="open",

    # 单只股票成交量限制
    volume_limit=0.025,
)


# =====================================================
# 11. 回测后统计：换手率、交易成本损耗、最大回撤区间
# =====================================================

def _get_raw_perf_df(performance):
    """
    兼容不同 BigQuant 环境下的 raw_perf 形式。

    有些环境：
        performance.raw_perf 是 DataFrame

    有些环境：
        performance.raw_perf 是对象，需要 read_df()
    """
    raw_perf_obj = performance.raw_perf

    if isinstance(raw_perf_obj, pd.DataFrame):
        return raw_perf_obj.copy()

    if hasattr(raw_perf_obj, "read_df"):
        return raw_perf_obj.read_df().copy()

    if isinstance(raw_perf_obj, dict):
        return pd.DataFrame(raw_perf_obj).copy()

    raise TypeError(
        f"无法识别 performance.raw_perf 的类型：{type(raw_perf_obj)}。"
        "请先运行 print(type(performance.raw_perf)) 和 print(performance.raw_perf) 检查。"
    )


def _as_list(x):
    """
    将 raw_perf 中的 transactions / orders 字段统一转成 list。
    """
    if x is None:
        return []

    if isinstance(x, float) and pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    if isinstance(x, tuple):
        return list(x)

    if isinstance(x, dict):
        return [x]

    if isinstance(x, str):
        s = x.strip()

        if s == "" or s.lower() in ["nan", "none", "null", "[]"]:
            return []

        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, list):
                return obj
            if isinstance(obj, dict):
                return [obj]
            return []
        except Exception:
            return []

    return []


def _get_first_value(d, keys, default=np.nan):
    """
    从交易记录 dict 中兼容读取字段。
    """
    if not isinstance(d, dict):
        return default

    for key in keys:
        if key in d and d[key] is not None:
            return d[key]

    return default


def _to_float(x, default=np.nan):
    try:
        if pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def _find_portfolio_value_col(raw_perf_df, capital_base):
    """
    自动寻找账户总资产字段。
    如果找不到，但存在 algorithm_period_return，则估算 portfolio_value。
    """
    candidate_cols = [
        "portfolio_value",
        "account_value",
        "total_value",
        "ending_value",
        "net_value",
        "portfolio_value_",
    ]

    for col in candidate_cols:
        if col in raw_perf_df.columns:
            return col

    if "algorithm_period_return" in raw_perf_df.columns:
        raw_perf_df["_portfolio_value_estimated"] = (
            capital_base * (1.0 + raw_perf_df["algorithm_period_return"].astype(float))
        )
        return "_portfolio_value_estimated"

    raise ValueError(
        "raw_perf 中找不到账户总资产字段。请先运行：\n"
        "print(raw_perf_df.columns.tolist())\n"
        "查看实际字段名。"
    )


def _parse_transaction_records(raw_perf_df):
    """
    将 raw_perf 中的 transactions 或 orders 展平成交易明细。
    优先使用 transactions，如果没有则尝试 orders。
    """

    if "transactions" in raw_perf_df.columns:
        record_col = "transactions"
    elif "orders" in raw_perf_df.columns:
        record_col = "orders"
    else:
        return pd.DataFrame(
            columns=[
                "date",
                "instrument",
                "amount",
                "price",
                "side",
                "trade_value",
                "transaction_cost",
                "cost_source",
            ]
        )

    records = []

    for dt, row in raw_perf_df.iterrows():
        tx_list = _as_list(row.get(record_col, []))

        for tx in tx_list:
            if not isinstance(tx, dict):
                continue

            instrument = _get_first_value(
                tx,
                ["instrument", "sid", "symbol", "asset", "order_book_id", "code"],
                default=""
            )

            amount = _to_float(
                _get_first_value(
                    tx,
                    ["amount", "qty", "quantity", "filled", "filled_quantity", "volume"],
                    default=0.0
                ),
                default=0.0
            )

            price = _to_float(
                _get_first_value(
                    tx,
                    ["price", "fill_price", "filled_price", "avg_price"],
                    default=np.nan
                ),
                default=np.nan
            )

            explicit_value = _to_float(
                _get_first_value(
                    tx,
                    ["trade_value", "turnover", "value", "money", "amount_value"],
                    default=np.nan
                ),
                default=np.nan
            )

            if pd.notna(explicit_value):
                trade_value = abs(explicit_value)
            elif pd.notna(price):
                trade_value = abs(amount * price)
            else:
                trade_value = np.nan

            side_raw = _get_first_value(
                tx,
                ["side", "direction", "action", "transaction_side"],
                default=""
            )

            side_str = str(side_raw).lower()

            if side_str in ["buy", "long", "open", "买入"]:
                side = "buy"
            elif side_str in ["sell", "short", "close", "卖出"]:
                side = "sell"
            else:
                side = "buy" if amount > 0 else "sell"

            actual_cost_candidates = [
                "commission",
                "transaction_cost",
                "cost",
                "fee",
                "fees",
                "tax",
                "stamp_tax",
                "total_cost",
            ]

            actual_cost_values = []
            for key in actual_cost_candidates:
                val = _to_float(tx.get(key, np.nan), default=np.nan)
                if pd.notna(val):
                    actual_cost_values.append(abs(val))

            if len(actual_cost_values) > 0:
                transaction_cost = sum(actual_cost_values)
                cost_source = "raw_perf"
            else:
                if pd.notna(trade_value):
                    rate = BUY_COST if side == "buy" else SELL_COST
                    transaction_cost = max(trade_value * rate, MIN_COST)
                    cost_source = "estimated"
                else:
                    transaction_cost = np.nan
                    cost_source = "unknown"

            records.append({
                "date": dt,
                "instrument": instrument,
                "amount": amount,
                "price": price,
                "side": side,
                "trade_value": trade_value,
                "transaction_cost": transaction_cost,
                "cost_source": cost_source,
            })

    tx_df = pd.DataFrame(records)

    if not tx_df.empty:
        tx_df["date"] = pd.to_datetime(tx_df["date"])

    return tx_df


def _calc_drawdown_interval(raw_perf_df, portfolio_value_col):
    """
    计算最大回撤和最大回撤区间。
    """
    portfolio_value = raw_perf_df[portfolio_value_col].astype(float).dropna()

    if portfolio_value.empty:
        raise ValueError(f"{portfolio_value_col} 为空，无法计算最大回撤区间。")

    net_value = portfolio_value / portfolio_value.iloc[0]
    running_max = net_value.cummax()
    drawdown = net_value / running_max - 1.0

    max_dd_end = drawdown.idxmin()
    max_drawdown = drawdown.loc[max_dd_end]

    max_dd_start = net_value.loc[:max_dd_end].idxmax()
    peak_value = net_value.loc[max_dd_start]
    trough_value = net_value.loc[max_dd_end]

    after_trough = net_value.loc[max_dd_end:]
    recovered = after_trough[after_trough >= peak_value]

    if len(recovered) > 0:
        recovery_date = recovered.index[0]
        recovered_flag = True
    else:
        recovery_date = pd.NaT
        recovered_flag = False

    return {
        "最大回撤": max_drawdown,
        "最大回撤开始日": max_dd_start,
        "最大回撤结束日": max_dd_end,
        "最大回撤修复日": recovery_date,
        "是否已修复": recovered_flag,
        "回撤峰值净值": peak_value,
        "回撤谷底净值": trough_value,
    }


def analyze_backtest_extra(performance, capital_base):
    """
    输出：
    1. 策略换手率
    2. 交易成本损耗
    3. 最大回撤区间
    """

    raw_perf = _get_raw_perf_df(performance)

    # 处理日期索引
    if not isinstance(raw_perf.index, pd.DatetimeIndex):
        if "period_open" in raw_perf.columns:
            raw_perf.index = pd.to_datetime(raw_perf["period_open"])
        elif "date" in raw_perf.columns:
            raw_perf.index = pd.to_datetime(raw_perf["date"])
        else:
            raw_perf.index = pd.to_datetime(raw_perf.index)

    raw_perf = raw_perf.sort_index()

    print("raw_perf 字段列表：")
    print(raw_perf.columns.tolist())

    portfolio_value_col = _find_portfolio_value_col(
        raw_perf,
        capital_base=capital_base
    )

    tx_df = _parse_transaction_records(raw_perf)

    portfolio_value = raw_perf[portfolio_value_col].astype(float)

    if tx_df.empty:
        total_trade_value = 0.0
        total_transaction_cost = 0.0
        trade_days = 0
        avg_daily_turnover = 0.0
        avg_trade_day_turnover = 0.0
        total_turnover = 0.0
        annualized_turnover = 0.0
        cost_source = "no_transaction_records"
    else:
        tx_df["trade_value"] = tx_df["trade_value"].fillna(0.0)
        tx_df["transaction_cost"] = tx_df["transaction_cost"].fillna(0.0)

        daily_trade_value = tx_df.groupby("date")["trade_value"].sum()
        daily_transaction_cost = tx_df.groupby("date")["transaction_cost"].sum()

        daily_stat = pd.DataFrame(index=raw_perf.index)
        daily_stat["portfolio_value"] = portfolio_value
        daily_stat["trade_value"] = daily_trade_value.reindex(daily_stat.index).fillna(0.0)
        daily_stat["transaction_cost"] = daily_transaction_cost.reindex(daily_stat.index).fillna(0.0)

        daily_stat["daily_turnover"] = (
            daily_stat["trade_value"] / daily_stat["portfolio_value"]
        ).replace([np.inf, -np.inf], np.nan).fillna(0.0)

        total_trade_value = daily_stat["trade_value"].sum()
        total_transaction_cost = daily_stat["transaction_cost"].sum()

        avg_portfolio_value = daily_stat["portfolio_value"].mean()

        total_turnover = (
            total_trade_value / avg_portfolio_value
            if avg_portfolio_value != 0
            else np.nan
        )

        days = len(daily_stat)
        years = days / 252
        annualized_turnover = total_turnover / years if years > 0 else np.nan

        avg_daily_turnover = daily_stat["daily_turnover"].mean()

        trade_days = int((daily_stat["trade_value"] > 0).sum())

        if trade_days > 0:
            avg_trade_day_turnover = daily_stat.loc[
                daily_stat["trade_value"] > 0,
                "daily_turnover"
            ].mean()
        else:
            avg_trade_day_turnover = 0.0

        cost_source = ",".join(
            sorted(tx_df["cost_source"].dropna().unique().tolist())
        )

    drawdown_info = _calc_drawdown_interval(
        raw_perf,
        portfolio_value_col=portfolio_value_col
    )

    final_portfolio_value = float(portfolio_value.iloc[-1])

    total_cost_over_initial = (
        total_transaction_cost / capital_base
        if capital_base != 0
        else np.nan
    )

    total_cost_over_final = (
        total_transaction_cost / final_portfolio_value
        if final_portfolio_value != 0
        else np.nan
    )

    summary = pd.DataFrame([
        ["总成交额", total_trade_value, "买入成交额 + 卖出成交额，按绝对值加总"],
        ["总换手率", total_turnover, "总成交额 / 回测期平均资产"],
        ["年化换手率", annualized_turnover, "总换手率 / 回测年数"],
        ["平均日换手率", avg_daily_turnover, "每日成交额 / 当日资产，然后取均值"],
        ["有交易日平均换手率", avg_trade_day_turnover, "只在有成交的日期上取平均"],
        ["有交易日期数", trade_days, "发生实际交易的日期数量"],
        ["交易成本合计", total_transaction_cost, f"成本来源：{cost_source}"],
        ["交易成本 / 初始资金", total_cost_over_initial, "交易成本合计 / 初始资金"],
        ["交易成本 / 期末资产", total_cost_over_final, "交易成本合计 / 期末资产"],
        ["最大回撤", drawdown_info["最大回撤"], "净值从峰值到谷底的最大跌幅"],
        ["最大回撤开始日", drawdown_info["最大回撤开始日"], "最大回撤对应的峰值日期"],
        ["最大回撤结束日", drawdown_info["最大回撤结束日"], "最大回撤对应的谷底日期"],
        ["最大回撤修复日", drawdown_info["最大回撤修复日"], "净值重新回到前高的日期；NaT表示截至回测结束未修复"],
        ["最大回撤是否已修复", drawdown_info["是否已修复"], ""],
    ], columns=["指标", "数值", "说明"])

    display_summary = summary.copy()

    percent_items = [
        "总换手率",
        "年化换手率",
        "平均日换手率",
        "有交易日平均换手率",
        "交易成本 / 初始资金",
        "交易成本 / 期末资产",
        "最大回撤",
    ]

    money_items = [
        "总成交额",
        "交易成本合计",
    ]

    def _format_value(row):
        name = row["指标"]
        value = row["数值"]

        if pd.isna(value):
            return ""

        if name in percent_items:
            return f"{float(value):.2%}"

        if name in money_items:
            return f"{float(value):,.2f}"

        if name in ["最大回撤开始日", "最大回撤结束日", "最大回撤修复日"]:
            try:
                return pd.to_datetime(value).strftime("%Y-%m-%d")
            except Exception:
                return str(value)

        if isinstance(value, (bool, np.bool_)):
            return "是" if value else "否"

        if isinstance(value, (int, np.integer)):
            return f"{value}"

        if isinstance(value, (float, np.floating)):
            return f"{value:.6f}"

        return str(value)

    display_summary["数值"] = display_summary.apply(_format_value, axis=1)

    print("\n" + "=" * 100)
    print("回测补充统计：换手率、交易成本损耗、最大回撤区间")
    print("=" * 100)

    display(display_summary)

    if not tx_df.empty:
        tx_preview = tx_df.head(20).copy()

        for col in ["trade_value", "transaction_cost"]:
            tx_preview[col] = tx_preview[col].map(
                lambda x: f"{x:,.2f}" if pd.notna(x) else ""
            )

        print("\n交易明细样例：")
        display(tx_preview)

    else:
        print("\n没有在 raw_perf 中解析到 transactions 或 orders 交易明细。")
        print("如果你想进一步计算真实换手和成本，需要检查 raw_perf 字段中是否存在交易记录字段。")

    return raw_perf, tx_df, summary


raw_perf_df, transaction_df, extra_summary = analyze_backtest_extra(
    performance=performance,
    capital_base=CAPITAL_BASE
)

通过这个策略一共运行了三段时间的回测，回测结果显示，该策略在 2022—2024 区间、2024—2026 区间表现优异，夏普比率分别可以达到 0.67 和 1.06，最大回撤分别为 21.46%、15.83%，并且都能跑赢基准收益。但在 2020—2022 区间，策略在一个时间段内没有跑赢基准收益。对此，结合当时具体背景，做出如下分析：

首先，从因子本身的经济含义来看，exp_wgt_return_6m 并不是单纯追逐过去涨幅较高股票的传统动量因子，而是对过去 6 个月的日收益率进行“换手率加权 + 指数衰减加权”后的结果。在实际选股中，本策略按照该因子值升序排列，买入因子值较低的股票，因此其本质更接近于一种小市值股票池中的中短期反转策略：即寻找过去一段时间内，尤其是近期高换手背景下表现较弱、可能存在超跌修复机会的股票。华泰研报中也指出，exp_wgt_return_6m 属于表现较突出的改进动量类因子，其构造方式是在最近 N 个月内以每日换手率乘以指数衰减函数作为权重，对每日收益率求加权平均。

2020—2022 年期间，该策略表现相对一般，主要原因在于当时市场环境并不完全适合这类反转型因子发挥作用。2020 年初新冠疫情爆发后，市场首先经历了一轮剧烈的系统性冲击，随后虽然出现流动性宽松和风险偏好修复，但行情结构并不是简单的普涨或轮动，而是明显偏向于“核心资产抱团”和“强者恒强”。消费、医药、新能源、部分科技龙头等确定性较高的方向持续受到资金追捧，而大量小市值、弱势股并没有同步修复。在这种市场结构下，过去一段时间表现较弱的股票未必会迅速反弹，反而可能继续跑输强势板块，因此以“低因子值股票”为主要持仓对象的策略容易阶段性落后于基准。

其次，疫情期间股票下跌往往不只是短期情绪过度反应，而可能对应真实的基本面压力。部分线下消费、制造、交通运输、地产链、供应链相关公司受到经营活动受限、需求收缩、现金流恶化等因素影响，其股价下跌具有基本面重估的成分。在这种情况下，策略容易把“基本面恶化导致的下跌”误判为“短期超跌后的反转机会”，从而买入一些继续承压的股票。也就是说，反转因子最适合捕捉的是市场情绪或流动性冲击导致的短期错杀，但在疫情这种基本面冲击较强的阶段，因子的有效性会明显下降。

再次，2021 年至 2022 年上半年，市场还叠加了海外流动性收紧、成长股估值压缩、俄乌冲突、国内疫情反复等多重扰动。尤其是 2022 年，市场风险偏好显著下降，小市值股票对流动性和风险偏好的变化更加敏感。虽然本策略中加入了中证 1000 指数 20 日均线作为趋势过滤条件，在弱势市场中会降低总仓位，但由于防御仓位仍保留 30%，且均线类趋势信号本身具有一定滞后性，因此在快速下跌或反复震荡的行情中，策略仍然会承受一定回撤，难以完全规避系统性风险。

而在 2022 年之后，该策略表现明显改善，说明市场环境逐渐转向更适合该因子的状态。一方面，沪深 300 等大盘核心资产整体表现偏弱，过去占优的核心资产抱团逻辑瓦解；另一方面，小盘股、题材股、超跌股和轮动行情变得更加活跃。在这种环境中，前期跌幅较大、成交较活跃、筹码经历过充分换手的小市值股票，更容易在资金回流时出现修复性上涨。因此，exp_wgt_return_6m 所捕捉的“高换手弱势后的反转效应”开始发挥作用，策略也更容易跑赢基准。

综合来看，该策略更适用于小盘股相对活跃、市场风格偏轮动、指数处于震荡或震荡上行、个股存在超跌修复机会的行情；而在核心资产抱团、强趋势延续、系统性风险冲击、基本面被集中下修的行情中，策略表现可能会阶段性走弱。因此，2020—2022 年回测表现一般，并不代表因子完全失效，而是说明该因子具有明显的市场环境依赖性：当市场主要由宏观冲击、行业景气分化和强趋势风格主导时，反转型动量因子的效果会被压制；当市场进入小盘轮动和超跌修复阶段后，该因子的优势才会更加明显。


## 1. exp_wgt_return_6m 因子的构建逻辑

exp_wgt_return_6m 是华泰金工研报中表现较突出的改进动量类因子之一。它的核心思想是：在过去 6 个月内，对个股每日收益率进行加权平均，权重同时考虑两个因素：日换手率和指数衰减权重。也就是说，距离当前截面日越近、成交越活跃的收益率，对最终因子值的影响越大。

这个因子可以理解为：

近期高换手状态下，股票价格表现强弱的综合刻画。

但在本策略中，并不是买入因子值最高的股票，而是按照因子值升序排列，买入因子值最低的股票。因此，这个策略实际利用的不是传统意义上的“强者恒强动量”，而更接近于：

高换手弱势股的中短期反转效应。

也就是说，策略试图寻找那些过去 6 个月内，尤其是近期成交活跃但股价表现较弱的股票，认为这类股票在筹码充分换手、悲观预期释放后，未来可能出现修复性上涨。

## 2. 策略的具体构建方法

本策略并不是单纯使用 exp_wgt_return_6m 一个因子进行选股，而是在因子基础上叠加了股票池约束、行业过滤、持仓数量控制和趋势择时机制。根据你的代码，策略主要包括以下几个步骤：

选股池构建

策略首先对全市场股票进行基础过滤，剔除：

ST / 风险警示股票；
停牌股票；
上市时间过短的股票；
北交所股票；
部分低弹性、防御性或不适合该策略的行业。

随后，策略进一步限定在总市值排名后 1/3 的小市值股票池中选股。因此，该策略天然带有较强的小市值风格暴露。

因子排序与持仓构建

在候选股票池内，策略按照 exp_wgt_return_6m 因子值从低到高排序，选择排名靠前的 100 只股票作为目标持仓，并每隔 20 个交易日重新选股一次。

因此，该策略的核心选股逻辑可以概括为：

在小市值股票中，买入过去一段时间内高换手但表现较弱、可能存在超跌修复机会的股票。

趋势择时与仓位控制

策略还引入了中证 1000 指数 20 日均线作为趋势过滤条件：

当中证 1000 指数位于 20 日均线上方时，认为小盘股市场环境较强，目标总仓位提高至接近满仓；
当中证 1000 指数位于 20 日均线下方时，认为市场环境较弱，目标总仓位降低至防御仓位。

所以，这个策略并不只是单因子选股策略，而是一个综合了：

小市值风格
反转型动量因子
行业过滤
中证 1000 趋势择时
分散持仓与定期调仓

的组合策略。

## 3. 这个因子和策略适用的市场环境
适用行情一：小盘股活跃的市场

该策略最适合小盘股相对占优的行情。由于策略股票池本身限定在总市值后 1/3 的股票中，所以它的收益来源很大一部分依赖于小市值股票是否具备相对强势。

当市场资金愿意向小盘股、题材股、弹性资产扩散时，策略更容易获得超额收益。典型表现包括：

中证 1000、中证 2000 强于沪深 300；
小盘股成交活跃；
市场赚钱效应从大盘核心资产扩散到中小市值股票；
资金更愿意参与题材轮动和超跌修复。

在这种环境下，策略买入的“高换手弱势小票”更容易获得资金回流，从而产生反弹收益。

适用行情二：震荡市或震荡偏强市

这个策略并不一定最适合单边大牛市，反而更适合指数震荡、但个股机会活跃的市场。

在震荡市中，市场缺乏长期单一主线，资金往往会在不同板块、不同题材、不同市值区间之间快速轮动。此时，前期跌幅较大、筹码经过充分换手的股票，往往容易在某个阶段被资金重新挖掘。

因此，策略最理想的市场状态是：

指数没有明显单边下跌风险；
市场仍然存在较高交易活跃度；
板块轮动频繁；
超跌股、低位股、补涨股有持续表现机会。

在这种环境中，exp_wgt_return_6m 的反转效应更容易发挥作用。

适用行情三：市场存在“过度反应—修复”的阶段

这个因子本质上是在寻找过去表现较弱的股票，因此它依赖于一个前提：

股票过去的下跌或弱势，部分来自情绪、流动性、短期资金行为，而不是长期基本面恶化。

当市场因为短期恐慌、流动性冲击、资金踩踏或题材退潮导致部分股票过度下跌时，exp_wgt_return_6m 因子可能会识别出这些被错杀的股票。一旦市场风险偏好修复，这类股票就容易出现反弹。

所以，该策略适合的并不是所有下跌后的股票，而是适合捕捉：

情绪性下跌后的修复；
资金短期流出后的回补；
题材退潮后的二次轮动；
小盘股超跌后的补涨行情。
适用行情四：流动性相对宽松、风险偏好较高的市场

由于策略主要买入小市值股票，小盘股对市场流动性和风险偏好更加敏感。当市场流动性充裕、投资者风险偏好较高时，小盘股通常更容易获得估值弹性和交易弹性。

因此，该策略更适合：

市场成交额较高；
资金活跃度较强；
投资者愿意交易中小市值股票；
主题投资、题材轮动、短线资金活跃的阶段。

如果市场整体流动性不足，资金只愿意抱团少数确定性资产，那么该策略的表现可能会受到明显压制。

## 4. 这个因子和策略不适用的市场环境
不适用行情一：核心资产抱团、强者恒强行情

当市场处于核心资产抱团阶段时，资金会持续买入已经表现较强的行业龙头、机构重仓股和高景气资产。此时，市场逻辑是“强者继续强”，而不是“弱者反弹”。

但本策略买入的是过去表现较弱的股票，因此在这种环境中容易出现两个问题：

过早买入弱势股；
错过持续上涨的强势股。

这也是 2020—2021 年策略阶段性跑输基准的重要原因之一。当时市场更偏向消费、医药、新能源等核心资产和景气成长方向，而小市值弱势股的反转机会并不充分。

不适用行情二：系统性风险冲击行情

在疫情冲击、流动性危机、海外加息、地缘冲突等系统性风险环境中，股票下跌往往不是简单的情绪错杀，而是市场对盈利、估值和风险偏好的重新定价。

在这种环境下，过去跌得多的股票不一定会反弹，反而可能继续下跌。对于反转型策略来说，这种行情容易导致：

买入下跌趋势尚未结束的股票；
把基本面恶化误判为短期超跌；
在系统性下跌中承受较大回撤。

虽然本策略加入了中证 1000 趋势择时，但由于趋势信号存在滞后性，而且弱势环境下仍然保留一定防御仓位，因此无法完全规避系统性风险。

不适用行情三：基本面被持续下修的行业环境

如果某些股票下跌是因为行业景气恶化、盈利能力下降、商业模式受损或政策环境发生重大变化，那么这种下跌并不是短期错杀，而是基本面重估。

此时，exp_wgt_return_6m 可能会错误地把这类股票识别为“超跌反转机会”，但实际上这些股票可能是“价值陷阱”。

因此，该策略不适合在以下环境中过度依赖：

行业基本面持续恶化；
企业盈利预期连续下修；
政策或产业逻辑发生根本变化；
市场正在重新定价某类资产的长期价值。
不适用行情四：大盘蓝筹明显占优行情

由于本策略主要在小市值股票池中选股，如果市场风格明显偏向大盘蓝筹、高股息、央国企大票或机构重仓龙头，那么策略很容易跑输以沪深 300 为代表的大盘基准。

换句话说，当市场主线集中在大票，而不是小票时，该策略的小市值暴露反而会成为拖累。

## 5. 总体结论

综合来看，exp_wgt_return_6m 因子在本策略中发挥的并不是传统追涨动量作用，而是更接近于反转型动量因子。策略通过在小市值股票池中买入因子值较低的股票，试图捕捉“高换手弱势股”在情绪释放、筹码出清后的修复机会。

因此，该策略最适合的市场环境是：

小盘股相对大盘股更强；
市场处于震荡或震荡偏强状态；
板块和题材轮动活跃；
市场存在较多超跌修复机会；
流动性相对宽松，风险偏好较高；
下跌主要来自短期情绪和资金行为，而不是长期基本面恶化。

相反，该策略不适合的市场环境是：

核心资产抱团、强者恒强；
系统性风险集中释放；
小盘股流动性低迷；
行业基本面持续下修；
大盘蓝筹明显占优；
市场处于单边下跌或极端避险状态。

因此，对于这个因子和策略，不能简单评价为“长期有效”或“长期无效”。更准确的表述应该是：

该策略具有明显的市场环境依赖性。它的优势主要体现在小盘股活跃、市场轮动频繁、超跌修复机会较多的阶段；而在核心资产抱团、系统性风险冲击或基本面持续下修的阶段，策略可能阶段性失效。



## 市值分组回测

In [ ]:
# =====================================================
# exp_wgt_return_6m 因子策略：15 市值组 + 指定组别内因子最低 15% 等权
# 深度优化版：适用 BigQuant AIStudio / BigTrader 日频回测与提交模拟
#
# 优化重点：
# 1）只在低频调仓信号日计算并返回最终入选股票，避免全市场每日信号落地。
# 2）删除 m_lead / next_open / next_limit 等未来字段；交易限制改为在交易日读取当前 Bar 的 open/upper_limit/lower_limit。
# 3）信号日与交易日严格错开：T 日收盘后形成选股与趋势信号，T+1 日开盘执行。
# 4）使用 BigTrader 当前 Bar 撮合模式，配合 T+1 执行日信号，实现“无未来函数 + 开盘价成交约束”。
# 5）不再查询全量 trade_status_df，也不再把所有曾入选股票扩展成“全股票 × 全日期”的大表。
# 6）initialize 阶段将信号预索引成 date -> dict，handle_data 不再逐日扫描 DataFrame。
# 7）传入 BigTrader 的 data 仅保留交易必要字段，减少内存占用。
#
# 交易逻辑：
# - 市值组：每个交易日按 total_market_cap 从小到大分成 15 组；1 = 最小市值组，15 = 最大市值组。
# - SELECT_MCAP_GROUPS = [1, 2, 3] 表示只在第 1、2、3 组里选股。
# - 每个指定市值组内部选择 exp_wgt_return_6m 因子值最低的 15%。
# - 所有最终入选股票按趋势过滤后的总仓位等权配置。
# - 趋势仓位使用上一交易日收盘后的指数趋势信号，避免开盘交易使用当日收盘信息。
# - 买入时：当前开盘价 >= 当前涨停价则不买；卖出时：当前开盘价 <= 当前跌停价则不卖。
# =====================================================

import math
import hashlib
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

import dai
from bigquant import bigtrader


# =====================================================
# 1. 策略参数区
# =====================================================

START_DATE = "2023-01-01"
END_DATE = "2026-06-01"

# 市值分组数量：按 total_market_cap 从小到大分成 15 组
# 组号含义：1 = 最小市值组，15 = 最大市值组
MCAP_GROUP_COUNT = 15

# 指定参与选股的市值组，例如 [1, 2, 3] 表示只在最小的 3 个市值组中选股
SELECT_MCAP_GROUPS = [15,14]

# 每个指定市值组内，选择因子值最低的比例
GROUP_SELECT_PCT = 0.10

# 每隔多少个交易日重新选股一次
REBALANCE_DAYS = 20

# 强势市场目标总仓位
TARGET_TOTAL_WEIGHT = 0.98

# 弱势市场防御总仓位；如果想弱势时完全空仓，改成 0.0
DEFENSIVE_TOTAL_WEIGHT = 0.30

# 趋势过滤指数；小市值策略通常可使用中证1000
TREND_INDEX = "000852.SH"

# 指数趋势均线天数
TREND_MA_DAYS = 30

# exp_wgt_return_6m：6个月成交量加权衰减动量因子
FACTOR_MONTHS = 6
LOOKBACK_DAYS = 21 * FACTOR_MONTHS

# 为计算 m_lag 和指数均线，需要在回测开始日前多取一段历史
BEFORE_START_DAYS = max(300, TREND_MA_DAYS * 3)

# list_days 是自然日口径，不是交易日口径
MIN_LIST_DAYS = int(LOOKBACK_DAYS * 365 / 252) + 30

# 初始资金
CAPITAL_BASE = 1_000_000

# 剔除行业：使用中信一级行业 cs_level1_name
EXCLUDE_INDUSTRIES = [
    "交通运输",
    "电力及公用事业",
    "纺织服装",
    "轻工制造",
    "家电",
    "石油石化",
    "综合",
    "银行",
]

# 回测基准
BENCHMARK = "000300.SH"

# 是否打印每日调仓 / 风控日志
VERBOSE = False

# 手续费参数，需要和 context.set_commission 保持一致
BUY_COST = 0.0003
SELL_COST = 0.0013
MIN_COST = 5.0

# 是否使用当前 Bar 撮合。
# 本策略为了避免 m_lead 未来字段，并在 T+1 开盘根据 T 日信号执行，使用 CURRENT。
# 如果你的 BigTrader 环境不支持 set_vmatch_at，则代码会自动降级，但此时不建议用本文件做严格开盘限制回测。
USE_CURRENT_BAR_MATCHING = True

# 浮点比较容忍度
EPS = 1e-10


# =====================================================
# 2. 参数校验与辅助函数
# =====================================================

def _normalize_mcap_groups(groups, group_count):
    if groups is None or len(groups) == 0:
        raise ValueError("SELECT_MCAP_GROUPS 不能为空，例如 [1, 2, 3]。")

    normalized = sorted({int(g) for g in groups})
    invalid = [g for g in normalized if g < 1 or g > group_count]

    if invalid:
        raise ValueError(
            f"SELECT_MCAP_GROUPS 中存在非法组别 {invalid}；"
            f"有效范围为 1 到 {group_count}。"
        )

    return normalized


def _sql_quote(values):
    if values is None or len(values) == 0:
        raise ValueError("SQL 列表不能为空。")
    return ", ".join(["'" + str(x).replace("'", "''") + "'" for x in values])


def _sql_int_list(values):
    if values is None or len(values) == 0:
        raise ValueError("SQL 整数列表不能为空。")
    return ", ".join([str(int(x)) for x in values])


def lag(field: str, k: int) -> str:
    """生成 BigQuant DAI SQL 的 m_lag 表达式。k=0 表示当前值。"""
    if k == 0:
        return field
    return f"m_lag({field}, {k})"


def _safe_float(x, default=np.nan):
    try:
        if x is None or pd.isna(x):
            return default
        return float(x)
    except Exception:
        return default


def _safe_int(x, default=0):
    try:
        if x is None or pd.isna(x):
            return default
        return int(x)
    except Exception:
        return default


SELECT_MCAP_GROUPS = _normalize_mcap_groups(SELECT_MCAP_GROUPS, MCAP_GROUP_COUNT)

if not (0 < GROUP_SELECT_PCT <= 1):
    raise ValueError("GROUP_SELECT_PCT 必须位于 (0, 1] 区间内。")

if REBALANCE_DAYS <= 0:
    raise ValueError("REBALANCE_DAYS 必须为正整数。")

calc_start_date = (
    datetime.strptime(START_DATE, "%Y-%m-%d") - timedelta(days=BEFORE_START_DAYS)
).strftime("%Y-%m-%d")

industry_sql = _sql_quote(EXCLUDE_INDUSTRIES)
selected_mcap_group_sql = _sql_int_list(SELECT_MCAP_GROUPS)

print("策略参数：")
print(f"回测区间：{START_DATE} ~ {END_DATE}")
print(f"市值分组数量：{MCAP_GROUP_COUNT}；选定组别：{SELECT_MCAP_GROUPS}")
print(f"每组选取比例：{GROUP_SELECT_PCT:.2%}；调仓间隔：{REBALANCE_DAYS} 个交易日")
print(f"趋势指数：{TREND_INDEX}；趋势均线：MA{TREND_MA_DAYS}")


# =====================================================
# 3. 构造 exp_wgt_return_6m 因子表达式
# =====================================================

num_terms = []
den_terms = []

for i in range(LOOKBACK_DAYS):
    decay_weight = math.exp(-i / FACTOR_MONTHS / 4.0)

    close_i = lag("close", i)
    close_i_1 = lag("close", i + 1)
    turn_i = lag("turn", i)

    ret_i = f"(({close_i} / NULLIF({close_i_1}, 0)) - 1.0)"

    num_terms.append(f"COALESCE(({decay_weight:.12g} * {turn_i} * {ret_i}), 0.0)")
    den_terms.append(f"COALESCE(({decay_weight:.12g} * {turn_i}), 0.0)")

num_expr = " + ".join(num_terms)
den_expr = " + ".join(den_terms)


# =====================================================
# 4. 查询指数趋势与交易日历
#    注意：趋势信号在 date 收盘后才可得，因此实际交易使用上一交易日的趋势结果。
# =====================================================

index_trend_sql = f"""
WITH index_raw AS (
    SELECT
        date,
        close AS trend_index_close,
        AVG(close) OVER (
            PARTITION BY instrument
            ORDER BY date
            ROWS BETWEEN {TREND_MA_DAYS - 1} PRECEDING AND CURRENT ROW
        ) AS trend_index_ma
    FROM cn_stock_index_bar1d
    WHERE instrument = '{TREND_INDEX}'
      AND date >= '{calc_start_date}'
      AND date <= '{END_DATE}'
),
index_trend AS (
    SELECT
        date,
        trend_index_close,
        trend_index_ma,
        CASE
            WHEN trend_index_ma IS NULL THEN 0
            WHEN trend_index_close > trend_index_ma THEN 1
            ELSE 0
        END AS risk_on,
        CASE
            WHEN trend_index_ma IS NULL THEN {DEFENSIVE_TOTAL_WEIGHT}
            WHEN trend_index_close > trend_index_ma THEN {TARGET_TOTAL_WEIGHT}
            ELSE {DEFENSIVE_TOTAL_WEIGHT}
        END AS trend_total_weight
    FROM index_raw
)
SELECT
    date,
    trend_index_close,
    trend_index_ma,
    risk_on,
    trend_total_weight
FROM index_trend
WHERE date >= '{START_DATE}'
ORDER BY date ASC
"""

daily_trend_raw = dai.query(index_trend_sql).df()
daily_trend_raw["date"] = pd.to_datetime(daily_trend_raw["date"]).dt.strftime("%Y-%m-%d")

daily_trend_raw = daily_trend_raw.sort_values("date").reset_index(drop=True)

if daily_trend_raw.empty:
    raise ValueError("daily_trend_raw 为空，请检查趋势指数代码或日期范围。")

all_dates = daily_trend_raw["date"].tolist()

if len(all_dates) < 2:
    raise ValueError("交易日数量不足，无法形成 T 日信号、T+1 日交易。")

# T 日形成趋势信号，T+1 日开盘使用。
daily_trend_trade = daily_trend_raw.copy()
daily_trend_trade["trend_signal_date"] = daily_trend_trade["date"].shift(1)
for col in ["trend_index_close", "trend_index_ma", "risk_on", "trend_total_weight"]:
    daily_trend_trade[col] = daily_trend_trade[col].shift(1)

daily_trend_trade = daily_trend_trade.dropna(subset=["trend_signal_date", "trend_total_weight"]).copy()
daily_trend_trade["risk_on"] = daily_trend_trade["risk_on"].astype(np.int8)
daily_trend_trade["trend_total_weight"] = daily_trend_trade["trend_total_weight"].astype(np.float32)

# 低频选股信号日在收盘后产生，下一交易日开始执行。
rebalance_signal_dates = all_dates[::REBALANCE_DAYS]
next_date_map = {all_dates[i]: all_dates[i + 1] for i in range(len(all_dates) - 1)}
rebalance_signal_dates = [d for d in rebalance_signal_dates if d in next_date_map]

if len(rebalance_signal_dates) == 0:
    raise ValueError("rebalance_signal_dates 为空，请检查交易日期或 REBALANCE_DAYS。")

rebalance_signal_date_sql = _sql_quote(rebalance_signal_dates)

rebalance_schedule_df = pd.DataFrame({
    "rebalance_signal_date": rebalance_signal_dates,
    "effective_start_date": [next_date_map[d] for d in rebalance_signal_dates],
})

print("交易日期数量：", len(all_dates))
print("选股信号次数：", len(rebalance_signal_dates))
print("首个选股信号日：", rebalance_signal_dates[0], "；首个执行日：", rebalance_schedule_df["effective_start_date"].iloc[0])
print("最后选股信号日：", rebalance_signal_dates[-1], "；最后执行起始日：", rebalance_schedule_df["effective_start_date"].iloc[-1])


# =====================================================
# 5. 仅在调仓信号日查询最终目标股票
#    SQL 侧完成：基础过滤、市值分组、指定组别过滤、组内因子排序、组内最低 15% 选择。
# =====================================================

signal_sql = f"""
WITH factor_base AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        c_pct_rank(total_market_cap, ascending := true) AS mcap_pct,
        ({num_expr}) / NULLIF(({den_expr}), 0) AS exp_wgt_return_6m
    FROM cn_stock_prefactors
    WHERE date >= '{calc_start_date}'
      AND date <= '{END_DATE}'
      -- 只取沪深 A 股，剔除北交所
      AND list_sector IN (1, 2, 3)
      -- 剔除 ST / *ST / 风险警示
      AND is_risk_warning = 0
      -- 剔除停牌
      AND suspended = 0
      -- 剔除上市时间过短的新股
      AND list_days >= {MIN_LIST_DAYS}
      AND close IS NOT NULL
      AND turn IS NOT NULL
      AND total_market_cap IS NOT NULL
      AND cs_level1_name IS NOT NULL
),
with_mcap_group AS (
    SELECT
        *,
        CASE
            WHEN mcap_pct IS NULL THEN NULL
            WHEN mcap_pct >= 1.0 THEN {MCAP_GROUP_COUNT}
            ELSE CAST(FLOOR(mcap_pct * {MCAP_GROUP_COUNT}) + 1 AS INTEGER)
        END AS mcap_group
    FROM factor_base
),
universe AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        mcap_pct,
        mcap_group,
        exp_wgt_return_6m
    FROM with_mcap_group
    WHERE date IN ({rebalance_signal_date_sql})
      AND mcap_group IN ({selected_mcap_group_sql})
      AND cs_level1_name NOT IN ({industry_sql})
      AND exp_wgt_return_6m IS NOT NULL
),
ranked AS (
    SELECT
        date,
        instrument,
        cs_level1_name,
        total_market_cap,
        mcap_pct,
        mcap_group,
        exp_wgt_return_6m,
        COUNT(*) OVER (
            PARTITION BY date, mcap_group
        ) AS group_stock_count,
        ROW_NUMBER() OVER (
            PARTITION BY date, mcap_group
            ORDER BY exp_wgt_return_6m ASC, total_market_cap ASC, instrument ASC
        ) AS factor_rank_in_group
    FROM universe
),
selected AS (
    SELECT
        *,
        CAST(CEIL(group_stock_count * {GROUP_SELECT_PCT:.12g}) AS INTEGER) AS group_select_num
    FROM ranked
)
SELECT
    date AS rebalance_signal_date,
    instrument,
    cs_level1_name,
    total_market_cap,
    mcap_pct,
    mcap_group,
    exp_wgt_return_6m,
    group_stock_count,
    group_select_num,
    factor_rank_in_group
FROM selected
WHERE factor_rank_in_group <= group_select_num
ORDER BY rebalance_signal_date ASC, mcap_group ASC, factor_rank_in_group ASC, instrument ASC
"""

target_rebalance_df = dai.query(signal_sql).df()
target_rebalance_df["rebalance_signal_date"] = pd.to_datetime(
    target_rebalance_df["rebalance_signal_date"]
).dt.strftime("%Y-%m-%d")

if target_rebalance_df.empty:
    raise ValueError(
        "target_rebalance_df 为空，请检查日期范围、市值组别、行业过滤或因子字段。"
    )

target_rebalance_df = target_rebalance_df.sort_values(
    ["rebalance_signal_date", "mcap_group", "factor_rank_in_group", "instrument"]
).reset_index(drop=True)

# 降低内存占用：数值列下转，字符串列转 category。
for col in ["mcap_group", "group_stock_count", "group_select_num", "factor_rank_in_group"]:
    target_rebalance_df[col] = target_rebalance_df[col].astype(np.int32)
for col in ["total_market_cap", "mcap_pct", "exp_wgt_return_6m"]:
    target_rebalance_df[col] = target_rebalance_df[col].astype(np.float32)
for col in ["instrument", "cs_level1_name"]:
    target_rebalance_df[col] = target_rebalance_df[col].astype("category")

print("调仓信号日目标股票样例：")
print(target_rebalance_df.head(20))
print("调仓信号日数量：", target_rebalance_df["rebalance_signal_date"].nunique())
print("曾经入选目标池股票数量：", target_rebalance_df["instrument"].nunique())


# =====================================================
# 6. 构造每日目标持仓
#    T 日选股信号从 T+1 日开始执行；趋势仓位也使用 T 日收盘后可得信号。
# =====================================================

trade_dates = daily_trend_trade["date"].tolist()
effective_starts = rebalance_schedule_df["effective_start_date"].tolist()
effective_start_arr = np.array(effective_starts, dtype=object)

trade_schedule_df = pd.DataFrame({"date": trade_dates})
active_idx = np.searchsorted(effective_start_arr, trade_schedule_df["date"].values, side="right") - 1
trade_schedule_df = trade_schedule_df.loc[active_idx >= 0].copy()
trade_schedule_df["rebalance_signal_date"] = np.array(rebalance_signal_dates, dtype=object)[active_idx[active_idx >= 0]]

# 只保留可交易区间内的目标股票，不生成“所有曾入选股票 × 所有日期”的大表。
daily_target_df = trade_schedule_df.merge(
    target_rebalance_df,
    on="rebalance_signal_date",
    how="inner",
)

daily_target_df = daily_target_df.merge(
    daily_trend_trade[
        [
            "date",
            "trend_signal_date",
            "trend_index_close",
            "trend_index_ma",
            "risk_on",
            "trend_total_weight",
        ]
    ],
    on="date",
    how="left",
)

if daily_target_df.empty:
    raise ValueError("daily_target_df 为空，请检查每日目标构造逻辑。")

if daily_target_df[["risk_on", "trend_total_weight"]].isna().any().any():
    raise ValueError("daily_target_df 中存在缺失的趋势信号，请检查指数趋势数据。")

# 所有最终入选股票等权配置；趋势切到防御仓位时等比例降低所有股票目标权重。
daily_target_df["stock_count"] = daily_target_df.groupby("date", observed=True)["instrument"].transform("count")
daily_target_df["weight"] = daily_target_df["trend_total_weight"] / daily_target_df["stock_count"]

# 传给 BigTrader 的字段越少越快；调试字段可保留，但不传大字段。
signal_df = daily_target_df[
    [
        "date",
        "instrument",
        "weight",
        "rebalance_signal_date",
        "trend_signal_date",
        "risk_on",
        "trend_total_weight",
        "mcap_group",
        "factor_rank_in_group",
        "exp_wgt_return_6m",
    ]
].copy()

signal_df["instrument"] = signal_df["instrument"].astype(str)
signal_df["date"] = signal_df["date"].astype(str)
signal_df["rebalance_signal_date"] = signal_df["rebalance_signal_date"].astype(str)
signal_df["trend_signal_date"] = signal_df["trend_signal_date"].astype(str)
signal_df["risk_on"] = signal_df["risk_on"].astype(np.int8)
signal_df["mcap_group"] = signal_df["mcap_group"].astype(np.int16)
signal_df["factor_rank_in_group"] = signal_df["factor_rank_in_group"].astype(np.int32)
signal_df["weight"] = signal_df["weight"].astype(np.float32)
signal_df["trend_total_weight"] = signal_df["trend_total_weight"].astype(np.float32)
signal_df["exp_wgt_return_6m"] = signal_df["exp_wgt_return_6m"].astype(np.float32)

signal_df = signal_df.sort_values(
    ["date", "mcap_group", "factor_rank_in_group", "instrument"]
).reset_index(drop=True)

if signal_df.empty:
    raise ValueError("signal_df 为空。")

all_target_instruments = sorted(signal_df["instrument"].unique().tolist())

print("每日目标仓位样例：")
print(signal_df.head(30))
print("每日风控日期数量：", signal_df["date"].nunique())
print("进入 BigTrader 的股票数量：", len(all_target_instruments))
print("强势执行日数量：", int((signal_df.drop_duplicates("date")["risk_on"] == 1).sum()))
print("防御执行日数量：", int((signal_df.drop_duplicates("date")["risk_on"] == 0).sum()))

check_df = signal_df[
    [
        "date",
        "rebalance_signal_date",
        "trend_signal_date",
        "instrument",
        "mcap_group",
        "factor_rank_in_group",
        "risk_on",
        "trend_total_weight",
        "weight",
    ]
].copy()

signal_hash = hashlib.md5(
    check_df.to_csv(index=False).encode("utf-8")
).hexdigest()
print("信号哈希：", signal_hash)


# =====================================================
# 7. BigTrader 回测函数：无 DataFrame 日内扫描版本
# =====================================================

def _current_bar_values(data: bigtrader.IBarData, instrument: str):
    """读取当前 Bar 的开盘价、涨停价、跌停价、成交量。读取失败表示当前不可交易。"""
    try:
        row = data.current(instrument, ["open", "upper_limit", "lower_limit", "volume"])
    except Exception:
        return None

    try:
        open_price = _safe_float(row["open"])
        upper_limit = _safe_float(row["upper_limit"])
        lower_limit = _safe_float(row["lower_limit"])
        volume = _safe_float(row["volume"], default=0.0)
    except Exception:
        return None

    if pd.isna(open_price) or pd.isna(upper_limit) or pd.isna(lower_limit):
        return None

    return {
        "open": open_price,
        "upper_limit": upper_limit,
        "lower_limit": lower_limit,
        "volume": volume,
    }


def _can_buy_at_current_open(data: bigtrader.IBarData, instrument: str) -> bool:
    """当前开盘涨停或无有效成交数据时，不买入。"""
    bar = _current_bar_values(data, instrument)
    if bar is None:
        return False
    if bar["volume"] <= 0:
        return False
    return bar["open"] < bar["upper_limit"] - EPS


def _can_sell_at_current_open(data: bigtrader.IBarData, instrument: str) -> bool:
    """当前开盘跌停或无有效成交数据时，不卖出。"""
    bar = _current_bar_values(data, instrument)
    if bar is None:
        return False
    if bar["volume"] <= 0:
        return False
    return bar["open"] > bar["lower_limit"] + EPS


def _build_target_maps(signal_data: pd.DataFrame):
    """把 DataFrame 预索引成字典，避免 handle_data 每天做布尔筛选。"""
    target_by_date = {}
    meta_by_date = {}

    use_cols = ["instrument", "weight"]
    meta_cols = ["rebalance_signal_date", "trend_signal_date", "risk_on", "trend_total_weight"]

    for date, group in signal_data.groupby("date", sort=False):
        target_by_date[date] = dict(zip(group["instrument"].values, group["weight"].astype(float).values))
        first = group.iloc[0]
        meta_by_date[date] = {col: first[col] for col in meta_cols}

    return target_by_date, meta_by_date


def initialize(context: bigtrader.IContext):
    context.signal_data = context.data.copy()
    context.signal_data["date"] = context.signal_data["date"].astype(str)
    context.signal_data["instrument"] = context.signal_data["instrument"].astype(str)

    context.target_by_date, context.meta_by_date = _build_target_maps(context.signal_data)
    context.signal_dates = set(context.target_by_date.keys())

    # 记录上一次发出的目标权重，减少重复下单。
    # 若某日因涨跌停/停牌跳过，则不会更新该记录，后续会继续尝试。
    context.desired_weight_map = {}

    context.set_commission(
        bigtrader.PerOrder(
            buy_cost=BUY_COST,
            sell_cost=SELL_COST,
            min_cost=MIN_COST,
        )
    )

    if USE_CURRENT_BAR_MATCHING:
        try:
            vmatch_enum = getattr(bigtrader, "VMatchAt", None)
            if vmatch_enum is not None and hasattr(vmatch_enum, "CURRENT"):
                current_mode = vmatch_enum.CURRENT
            else:
                from bigtrader.constant import VMatchAt
                current_mode = VMatchAt.CURRENT
            context.set_vmatch_at(current_mode)
        except Exception as e:
            print("警告：当前 BigTrader 环境未能设置 VMatchAt.CURRENT。")
            print("原因：", repr(e))
            print("本策略无 m_lead 未来字段；若不能当前 Bar 撮合，则开盘涨跌停约束可能无法与成交时点完全对齐。")


def handle_data(context: bigtrader.IContext, data: bigtrader.IBarData):
    today = data.current_dt.strftime("%Y-%m-%d")

    target_map = context.target_by_date.get(today)
    if target_map is None:
        return

    meta = context.meta_by_date.get(today, {})

    if VERBOSE:
        print(
            f"{today} 执行：rebalance_signal_date={meta.get('rebalance_signal_date')}, "
            f"trend_signal_date={meta.get('trend_signal_date')}, "
            f"risk_on={meta.get('risk_on')}, "
            f"target_total_weight={float(meta.get('trend_total_weight', 0.0)):.2%}, "
            f"target_count={len(target_map)}"
        )

    current_positions = context.get_positions()
    holding_instruments = set(current_positions.keys())
    target_instruments = set(target_map.keys())

    # 1）先卖出已经不在目标池里的股票。
    # 如果当前开盘跌停或停牌卖不出，后续每天继续尝试。
    for instrument in sorted(holding_instruments - target_instruments):
        if not _can_sell_at_current_open(data, instrument):
            if VERBOSE:
                print(f"{today} 跳过卖出 {instrument}：当前开盘跌停、停牌或无有效行情")
            continue

        rv = context.order_target_percent(instrument, 0)
        if rv is None or rv >= 0:
            context.desired_weight_map[instrument] = 0.0
        elif VERBOSE:
            print(f"{today} 卖出 {instrument} 下单失败：rv={rv}")

    # 2）再处理目标股票：加仓/新开仓要求当前开盘未涨停；减仓要求当前开盘未跌停。
    for instrument, target_weight in target_map.items():
        target_weight = float(target_weight)
        prev_weight = float(context.desired_weight_map.get(instrument, 0.0))
        diff = target_weight - prev_weight

        if abs(diff) < 1e-8:
            continue

        if diff > 0:
            if not _can_buy_at_current_open(data, instrument):
                if VERBOSE:
                    print(f"{today} 跳过买入/加仓 {instrument}：当前开盘涨停、停牌或无有效行情")
                continue
        else:
            if not _can_sell_at_current_open(data, instrument):
                if VERBOSE:
                    print(f"{today} 跳过减仓 {instrument}：当前开盘跌停、停牌或无有效行情")
                continue

        rv = context.order_target_percent(instrument, target_weight)
        if rv is None or rv >= 0:
            context.desired_weight_map[instrument] = target_weight
        elif VERBOSE:
            print(f"{today} 调整 {instrument} 到 {target_weight:.4%} 下单失败：rv={rv}")


# =====================================================
# 8. 运行回测
# =====================================================

performance = bigtrader.run(
    market=bigtrader.Market.CN_STOCK,
    frequency=bigtrader.Frequency.DAILY,
    start_date=START_DATE,
    end_date=END_DATE,
    capital_base=CAPITAL_BASE,
    instruments=all_target_instruments,
    data=signal_df,
    initialize=initialize,
    handle_data=handle_data,
    benchmark=BENCHMARK,
    # T 日收盘后产生信号，T+1 日当前 Bar 开盘检查涨跌停并撮合。
    order_price_field_buy="open",
    order_price_field_sell="open",
    # 单只股票成交量限制
    volume_limit=0.025,
)
